# **Consistent Characters AI**

In [ ]:
# @title 1. Install dependencies (12 mins)
!pip install diffusers["torch"] transformers accelerate
!pip install git+https://github.com/huggingface/diffusers
!pip uninstall torch torchvision -y
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install --upgrade huggingface_hub
!git clone https://github.com/s0md3v/roop.git
%cd roop
!pip install -r requirements.txt
!pip install -q gdown onnxruntime-gpu mxnet-cu90==1.1.0
%cd ..
%cd ..
!pip install --upgrade numpy
!pip install "numpy<2"
!pip install pyngrok
!pip install reportlab
!pip install rembg
%cd /content/roop
import os
os.makedirs('models', exist_ok=True)
%cd models
# install & download in one go
!pip install -q gdown && gdown https://drive.google.com/uc?id=18To_aFTTQ5k4IIhUFxt7IssjQlZ6jWle
%cd /content/roop/roop
os.makedirs('models', exist_ok=True)
%cd models
# install & download in one go
!pip install -q gdown && gdown https://drive.google.com/uc?id=16lFZHv-aY9N3W91a7SJLgGgZFX7cTP4o
%cd /content
import io

# Path to the predictor you want to patch
predictor_path = "/content/roop/roop/predictor.py"

# New content that short‑circuits all checks
new_code = '''import threading
# keep imports so nothing breaks downstream
import numpy
import opennsfw2
from PIL import Image
from keras import Model

from roop.typing import Frame

# We disable the predictor entirely by never instantiating it.
PREDICTOR = None
THREAD_LOCK = threading.Lock()

def get_predictor() -> Model:
    # never called
    raise RuntimeError("NSFW predictor is disabled")

def clear_predictor() -> None:
    # no-op
    pass

def predict_frame(target_frame: Frame) -> bool:
    """
    Always returns False (never NSFW).
    """
    return False

def predict_image(target_path: str) -> bool:
    """
    Always returns False (never NSFW).
    """
    return False

def predict_video(target_path: str) -> bool:
    """
    Always returns False (never NSFW).
    """
    return False
'''

# Overwrite the file in one shot
with open(predictor_path, "w", encoding="utf-8") as f:
    f.write(new_code)

print(f"[✓] NSFW predictor disabled in {predictor_path}")



In [ ]:
# 🔁 Uninstall everything that could conflict
!pip uninstall -y torch torchvision torchaudio diffusers transformers accelerate huggingface_hub peft

# 🧱 Install compatible base libraries (PyTorch with CUDA 11.8)
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

# ✅ Install specific compatible versions
!pip install diffusers==0.27.2
!pip install transformers==4.39.3
!pip install accelerate==0.28.0  # this version has `clear_device_cache`
!pip install huggingface_hub==0.20.3
!pip install peft==0.10.0        # optional, for LoRA support
!pip install safetensors


# Optional (for face restoration)
!pip uninstall -y basicsr gfpgan
!pip install git+https://github.com/xinntao/BasicSR.git@master
!pip install git+https://github.com/TencentARC/GFPGAN.git@master



import torch
import torch.sparse._triton_ops_meta
print("✅ Triton ops are available – torch", torch.__version__)

In [ ]:
# @title 2. Import & configure pipelines (3 min)
import os
import shutil
import torch
from diffusers import StableDiffusionPipeline, AutoPipelineForInpainting

"""# Initialize both pipelines
pipe = StableDiffusionPipeline.from_pretrained(
    "SG161222/RealVisXL_V4.0",
    torch_dtype=torch.float16
).to("cuda")
pipe.safety_checker = None

"""
# Initialize both pipelines
pipe = StableDiffusionPipeline.from_pretrained(
    "redstonehero/lazymix_real_amateur_nudes_v30b",
    torch_dtype=torch.float16
).to("cuda")
pipe.safety_checker = None


# Initialize the inpainting pipeline
inpainting_pipeline = AutoPipelineForInpainting.from_pretrained(
    "Uminosachi/realisticVisionV51_v51VAE-inpainting",
    torch_dtype=torch.float16
).to("cuda")
inpainting_pipeline.safety_checker = None

# Roop directory
ROOP_DIR = "roop"


In [ ]:
# @title 2.2. Web UI
# === CONSTANT CHARACTER MAKER FLASK APP ===
from flask import Flask, request, redirect, url_for, render_template_string, send_from_directory, send_file, jsonify, Response
import glob, datetime, os, shutil, time, json
from werkzeug.utils import secure_filename
from pyngrok import ngrok
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas
from reportlab.lib import colors
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Image, Table, TableStyle, PageBreak
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
import zipfile
import cv2
import numpy as np
from PIL import Image, ImageOps, ImageDraw
from rembg import remove
import io
import torch
import queue
import threading
from collections import defaultdict
from google.colab import files  # Add this import for Colab file handling
# Add Google Drive integration
from google.colab import drive

from transformers import (
    BlipProcessor,
    BlipForConditionalGeneration,
    CLIPProcessor,
    CLIPModel,
    CLIPTokenizer
)
import torch.nn.functional as F

# Initialize BLIP and CLIP models for image prompt generation
blip_processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
blip_model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-large-patch14")
clip_model = CLIPModel.from_pretrained("openai/clip-vit-large-patch14")
clip_tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-large-patch14")

# -----------------------------------------------------------------------------
# 1) PARAMETERS
# -----------------------------------------------------------------------------
HEIGHT, WIDTH      = 1080, 720
STEPS, GUIDANCE    = 100, 7.5
NUM_SWAPS          = 2
ROOP_DIR           = "roop"  # Path to roop directory, adjust as needed

NEG_PROMPT = (
    "artificial, robotic, android-like, synthetic skin, glossy skin, plastic texture, "
    "3D render, uncanny valley, cropped, lowres, bad anatomy, bad hands, bad fingers, text, error, "
    "missing fingers, missing limbs, extra digit, extra fingers, extra hand, extra limbs, multiple arms, multiple legs, "
    "three hands, duplicated limbs, dislocated joints, broken limbs, fused limbs, malformed hands, malformed limbs, "
    "wrong limb count, anatomically incorrect, asymmetrical body, twisted pose, unnatural pose, distorted proportions, "
    "duplicates, worst quality, low quality, jpeg artifacts, signature, cropped body, watermark, blurry, bad feet, "
    "mutation, deformed, cross-eyed, malformed facial features, lopsided face, lazy eye, misshapen head, "
    "half body, cartoon, flat colors, childish drawing, sketch, painting style"
)


# -----------------------------------------------------------------------------
# 2) DIRECTORIES & PIPELINE
# -----------------------------------------------------------------------------
UPLOAD_FOLDER    = "faces"
GENERATED_FOLDER = "generated"
SWAPPED_FOLDER   = "swapped"
ARCHIVES_FOLDER  = "archives"
DRIVE_FOLDER     = "/content/drive/MyDrive/AI_Character_Generator"

for d in (UPLOAD_FOLDER, GENERATED_FOLDER, SWAPPED_FOLDER, ARCHIVES_FOLDER):
    os.makedirs(d, exist_ok=True)

# Add this function for Colab file upload
def upload_face():
    """Handle face upload in Colab environment"""
    uploaded = files.upload()
    if not uploaded:
        return None

    # Get the first uploaded file
    filename = list(uploaded.keys())[0]
    file_content = uploaded[filename]

    # Save the file
    face_path = os.path.join(UPLOAD_FOLDER, secure_filename(filename))
    with open(face_path, 'wb') as f:
        f.write(file_content)

    return secure_filename(filename)

# -----------------------------------------------------------------------------
# 3) FLASK + TEMPLATE
# -----------------------------------------------------------------------------
app = Flask(__name__)
app.config['UPLOAD_FOLDER'] = UPLOAD_FOLDER

# Function to save all data to Google Drive
def save_to_google_drive():
    """
    Create a backup of all generated content and save it to Google Drive
    Returns the path to the saved archive file
    """
    timestamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
    archive_name = f"ai_character_generator_backup_{timestamp}.zip"
    archive_path = os.path.join(ARCHIVES_FOLDER, archive_name)

    # Create a zip file containing all data folders
    with zipfile.ZipFile(archive_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        # Add faces folder
        for folder in [UPLOAD_FOLDER, GENERATED_FOLDER, SWAPPED_FOLDER]:
            for file in os.listdir(folder):
                file_path = os.path.join(folder, file)
                if os.path.isfile(file_path):
                    zipf.write(file_path, os.path.join(folder, file))

    # Ensure Google Drive is mounted
    try:
        # Create the destination folder in Google Drive if it doesn't exist
        os.makedirs(DRIVE_FOLDER, exist_ok=True)

        # Copy the archive to Google Drive
        drive_path = os.path.join(DRIVE_FOLDER, archive_name)
        shutil.copy2(archive_path, drive_path)

        # Create a marker file to identify the most recent backup
        with open(os.path.join(DRIVE_FOLDER, "latest_backup.txt"), 'w') as f:
            f.write(archive_name)

        return drive_path
    except Exception as e:
        print(f"Error saving to Google Drive: {str(e)}")
        return archive_path

# Function to restore data from Google Drive
def restore_from_google_drive():
    """
    Restore the most recent backup from Google Drive
    Returns True if successful, False otherwise
    """
    try:
        # Check if the latest backup marker exists
        marker_path = os.path.join(DRIVE_FOLDER, "latest_backup.txt")
        if not os.path.exists(marker_path):
            return False, "No backup found in Google Drive"

        # Read the name of the latest backup
        with open(marker_path, 'r') as f:
            latest_backup = f.read().strip()

        backup_path = os.path.join(DRIVE_FOLDER, latest_backup)
        if not os.path.exists(backup_path):
            return False, f"Backup file not found: {latest_backup}"

        # Extract the backup
        with zipfile.ZipFile(backup_path, 'r') as zipf:
            zipf.extractall(".")

        return True, f"Successfully restored backup: {latest_backup}"
    except Exception as e:
        return False, f"Error restoring from Google Drive: {str(e)}"

# Routes for Google Drive integration
@app.route('/save_to_drive')
def save_to_drive():
    """Route to save all data to Google Drive"""
    try:
        # Mount Google Drive if not already mounted
        drive.mount('/content/drive', force_remount=True)

        # Save data to Google Drive
        drive_path = save_to_google_drive()

        return jsonify({
            'success': True,
            'message': f'Successfully saved backup to Google Drive: {os.path.basename(drive_path)}'
        })
    except Exception as e:
        return jsonify({
            'success': False,
            'message': f'Error saving to Google Drive: {str(e)}'
        })

@app.route('/restore_from_drive')
def restore_from_drive():
    """Route to restore data from Google Drive"""
    try:
        # Mount Google Drive if not already mounted
        drive.mount('/content/drive', force_remount=True)

        # Restore data from Google Drive
        success, message = restore_from_google_drive()

        if success:
            return jsonify({
                'success': True,
                'message': message
            })
        else:
            return jsonify({
                'success': False,
                'message': message
            })
    except Exception as e:
        return jsonify({
            'success': False,
            'message': f'Error restoring from Google Drive: {str(e)}'
        })

# Add a route to check for backup at startup
@app.route('/check_for_backup')
def check_for_backup():
    """Check if a backup exists in Google Drive"""
    try:
        # Try to mount Google Drive
        try:
            drive.mount('/content/drive', force_remount=False)
        except:
            return jsonify({
                'exists': False,
                'message': 'Google Drive not mounted'
            })

        # Check if the backup folder exists
        if not os.path.exists(DRIVE_FOLDER):
            return jsonify({
                'exists': False,
                'message': 'No backup folder found'
            })

        # Check if the latest backup marker exists
        marker_path = os.path.join(DRIVE_FOLDER, "latest_backup.txt")
        if not os.path.exists(marker_path):
            return jsonify({
                'exists': False,
                'message': 'No backup marker found'
            })

        # Read the name of the latest backup
        with open(marker_path, 'r') as f:
            latest_backup = f.read().strip()

        backup_path = os.path.join(DRIVE_FOLDER, latest_backup)
        if not os.path.exists(backup_path):
            return jsonify({
                'exists': False,
                'message': f'Backup file not found: {latest_backup}'
            })

        return jsonify({
            'exists': True,
            'message': f'Found backup: {latest_backup}',
            'backup': latest_backup
        })
    except Exception as e:
        return jsonify({
            'exists': False,
            'message': f'Error checking for backup: {str(e)}'
        })

# Function to initialize Google Drive at startup
def initialize_google_drive():
    """Try to mount Google Drive when app starts"""
    try:
        # Try to mount Google Drive silently
        drive.mount('/content/drive', force_remount=False)
        print("Google Drive mounted successfully")
        return True
    except:
        print("Google Drive not mounted. You can mount it manually when needed.")
        return False

HTML = """
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>AI Character Generator</title>
    <link href="https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600&display=swap" rel="stylesheet">
    <style>
        :root {
            --primary-color: #4f46e5;
            --primary-dark: #4338ca;
            --secondary-color: #8b5cf6;
            --background-color: #f5f3ff;
            --card-background: #ffffff;
            --text-primary: #1e1b4b;
            --text-secondary: #6b7280;
            --border-radius: 16px;
            --spacing: 24px;
            --border-color: #e2e8f0;
            --shadow-color: rgba(0, 0, 0, 0.1);
        }

        [data-theme="dark"] {
            --primary-color: #818cf8;
            --primary-dark: #6366f1;
            --secondary-color: #a78bfa;
            --background-color: #1e1b4b;
            --card-background: #312e81;
            --text-primary: #f5f3ff;
            --text-secondary: #c4b5fd;
            --border-color: #4338ca;
            --shadow-color: rgba(0, 0, 0, 0.3);
        }

        * {
            margin: 0;
            padding: 0;
            box-sizing: border-box;
        }

        body {
            font-family: 'Inter', sans-serif;
            background-color: var(--background-color);
            color: var(--text-primary);
            line-height: 1.5;
            padding: var(--spacing);
            transition: background-color 0.3s ease, color 0.3s ease;
        }

        .container {
            max-width: 1400px;
            margin: 0 auto;
        }

        .nav {
            display: flex;
            justify-content: space-between;
            align-items: center;
            margin-bottom: calc(var(--spacing) * 2);
            padding: var(--spacing);
            background: var(--card-background);
            border-radius: var(--border-radius);
            box-shadow: 0 4px 6px var(--shadow-color);
        }

        .nav__title {
            font-size: 1.5rem;
            font-weight: 600;
            color: var(--primary-color);
        }

        .nav__actions {
            display: flex;
            gap: var(--spacing);
            align-items: center;
        }

        .theme-toggle {
            background: none;
            border: none;
            color: var(--text-primary);
            cursor: pointer;
            padding: 8px;
            border-radius: 50%;
            display: flex;
            align-items: center;
            justify-content: center;
            transition: background-color 0.2s ease;
        }

        .theme-toggle:hover {
            background-color: var(--border-color);
        }

        .main-content {
            display: grid;
            grid-template-columns: 1fr 2fr;
            gap: var(--spacing);
        }

        .control-panel {
            background: var(--card-background);
            padding: var(--spacing);
            border-radius: var(--border-radius);
            box-shadow: 0 4px 6px var(--shadow-color);
        }

        .gallery {
            background: var(--card-background);
            padding: var(--spacing);
            border-radius: var(--border-radius);
            box-shadow: 0 4px 6px var(--shadow-color);
        }

        .form-group {
            margin-bottom: var(--spacing);
        }

        .form-group label {
            display: block;
            margin-bottom: 8px;
            color: var(--text-primary);
            font-weight: 500;
        }

        .checkbox-label {
            display: flex;
            align-items: center;
            gap: 8px;
            cursor: pointer;
        }

        .checkbox-text {
            font-weight: 500;
        }

        .help-text {
            color: var(--text-secondary);
            font-size: 0.8rem;
            margin-top: 4px;
        }

        /* Reface mode styles */
        #refaceMode {
            width: 18px;
            height: 18px;
            accent-color: var(--primary-color);
        }

        .reface-active #standard-generation-fields {
            display: none;
        }

        .reface-active #reface-target-section {
            display: block !important;
            animation: fadeIn 0.3s ease;
        }

        /* Required field indicator */
        .form-group label.required::after {
            content: " *";
            color: #EF4444;
        }

        /* Update field indicators based on mode */
        .reface-active .face-required label::after {
            content: " *";
            color: #EF4444;
        }

        .reface-active #reface-target-section label::after {
            content: " *";
            color: #EF4444;
        }

        @keyframes fadeIn {
            from { opacity: 0; transform: translateY(-10px); }
            to { opacity: 1; transform: translateY(0); }
        }

        .form-control {
            width: 100%;
            padding: 12px;
            border: 1px solid var(--border-color);
            border-radius: calc(var(--border-radius) / 2);
            background: var(--card-background);
            color: var(--text-primary);
            transition: border-color 0.2s ease;
        }

        .form-control:focus {
            outline: none;
            border-color: var(--primary-color);
        }

        .btn {
            display: inline-flex;
            align-items: center;
            gap: 8px;
            padding: 12px 24px;
            border-radius: var(--border-radius);
            font-weight: 500;
            cursor: pointer;
            transition: all 0.2s ease;
            border: none;
        }

        .btn-primary {
            background: var(--primary-color);
            color: white;
        }

        .btn-primary:hover {
            background: var(--primary-dark);
            transform: translateY(-1px);
        }

        .btn-secondary {
            background: var(--secondary-color);
            color: white;
        }

        .btn-secondary:hover {
            background: var(--secondary-color);
            filter: brightness(0.9);
        }

        .gallery-grid {
            display: grid;
            grid-template-columns: repeat(auto-fill, minmax(300px, 1fr));
            gap: var(--spacing);
        }

        .gallery-item {
            position: relative;
            border-radius: var(--border-radius);
            overflow: hidden;
            box-shadow: 0 4px 6px var(--shadow-color);
            cursor: pointer;
        }

        .gallery-item__overlay {
            position: absolute;
            bottom: 0;
            left: 0;
            right: 0;
            padding: var(--spacing);
            background: linear-gradient(to top, rgba(0,0,0,0.8), transparent);
            color: white;
            transition: all 0.3s ease;
        }

        .gallery-item__title {
            font-weight: 600;
            margin-bottom: 4px;
        }

        .gallery-item__prompt {
            font-size: 0.875rem;
            opacity: 0.8;
            max-height: 60px;
            overflow: hidden;
            text-overflow: ellipsis;
            display: -webkit-box;
            -webkit-line-clamp: 2;
            -webkit-box-orient: vertical;
        }

        .gallery-item__actions {
            display: flex;
            justify-content: center;
            margin-top: 12px;
            opacity: 0;
            transform: translateY(10px);
            transition: all 0.3s ease;
        }

        .gallery-item:hover .gallery-item__actions {
            opacity: 1;
            transform: translateY(0);
        }

        .gallery-item__download {
            background: var(--primary-color);
            color: white;
            border: none;
            border-radius: 8px;
            padding: 8px 16px;
            font-size: 0.85rem;
            font-weight: 500;
            display: flex;
            align-items: center;
            gap: 6px;
            cursor: pointer;
            transition: all 0.2s;
            box-shadow: 0 2px 8px rgba(0, 0, 0, 0.25);
        }

        .gallery-item__download svg {
            width: 16px;
            height: 16px;
        }

        .gallery-item__download:hover {
            background: var(--primary-dark);
            transform: translateY(-2px);
            box-shadow: 0 4px 12px rgba(0, 0, 0, 0.3);
        }

        .gallery-item__delete {
            background: #EF4444;
            color: white;
            border: none;
            border-radius: 8px;
            padding: 8px 16px;
            font-size: 0.85rem;
            font-weight: 500;
            display: flex;
            align-items: center;
            gap: 6px;
            cursor: pointer;
            transition: all 0.2s;
            margin-left: 8px;
            box-shadow: 0 2px 8px rgba(0, 0, 0, 0.25);
        }

        .gallery-item__delete svg {
            width: 16px;
            height: 16px;
        }

        .gallery-item__delete:hover {
            background: #DC2626;
            transform: translateY(-2px);
            box-shadow: 0 4px 12px rgba(0, 0, 0, 0.3);
        }

        .gallery-item img {
            width: 100%;
            height: 300px;
            object-fit: cover;
            transition: transform 0.3s ease;
        }

        .gallery-item:hover img {
            transform: scale(1.05);
        }

        .modal {
            display: none;
            position: fixed;
            top: 0;
            left: 0;
            width: 100%;
            height: 100%;
            background-color: rgba(0, 0, 0, 0.9);
            z-index: 1000;
            opacity: 0;
            transition: opacity 0.3s ease;
        }

        .modal.active {
            display: flex;
            opacity: 1;
            align-items: center;
            justify-content: center;
        }

        .modal-content {
            position: relative;
            max-width: 90vw;
            max-height: 90vh;
            background: var(--card-background);
            border-radius: var(--border-radius);
            padding: 20px;
        }

        .modal-close {
            position: absolute;
            top: 10px;
            right: 10px;
            color: var(--text-primary);
            font-size: 24px;
            cursor: pointer;
            background: none;
            border: none;
            padding: 5px;
            z-index: 1002;
        }

        .modal-layout {
            display: flex;
            gap: 20px;
            max-width: 90vw;
            max-height: 90vh;
        }

        .modal-image {
            max-width: 100%;
            max-height: 80vh;
            object-fit: contain;
            border-radius: 8px;
            opacity: 0;
            transition: opacity 0.3s ease;
        }

        .modal-image.loaded {
            opacity: 1;
        }

        .modal-image-container {
            flex: 1;
            position: relative;
            max-width: 70%;
            display: flex;
            flex-direction: column;
            align-items: center;
            background: var(--background-color);
            border-radius: var(--border-radius);
            min-height: 300px;
        }

        .modal-image-loading {
            position: absolute;
            top: 50%;
            left: 50%;
            transform: translate(-50%, -50%);
            width: 40px;
            height: 40px;
            border: 4px solid var(--border-color);
            border-top: 4px solid var(--primary-color);
            border-radius: 50%;
            animation: spin 1s linear infinite;
        }

        .modal-info {
            flex: 0 0 300px;
            background: var(--card-background);
            padding: 20px;
            border-radius: var(--border-radius);
            overflow-y: auto;
            max-height: 90vh;
        }

        .modal-prompt {
            color: var(--text-primary);
            font-size: 1rem;
            line-height: 1.5;
            white-space: pre-wrap;
            padding: 15px;
            background: var(--background-color);
            border-radius: var(--border-radius);
            margin-top: 10px;
        }

        .modal-actions {
            position: absolute;
            top: 20px;
            left: 20px;
            display: flex;
            gap: 10px;
            z-index: 1001;
        }

        .modal-nav {
            position: absolute;
            top: 50%;
            transform: translateY(-50%);
            width: 100%;
            display: flex;
            justify-content: space-between;
            padding: 0 20px;
            z-index: 1001;
        }

        .modal-nav-btn {
            background: rgba(0, 0, 0, 0.5);
            color: white;
            border: none;
            border-radius: 50%;
            width: 40px;
            height: 40px;
            display: flex;
            align-items: center;
            justify-content: center;
            cursor: pointer;
            transition: background-color 0.2s ease;
        }

        .modal-nav-btn:hover {
            background: rgba(0, 0, 0, 0.8);
        }

        .modal-nav-btn:disabled {
            opacity: 0.5;
            cursor: not-allowed;
        }

        .modal-counter {
            position: absolute;
            top: 20px;
            right: 60px;
            background: rgba(0, 0, 0, 0.5);
            color: white;
            padding: 5px 10px;
            border-radius: 15px;
            font-size: 0.875rem;
        }

        .loading-overlay {
            position: fixed;
            top: 0;
            left: 0;
            width: 100%;
            height: 100%;
            background: rgba(0, 0, 0, 0.85);
            backdrop-filter: blur(8px);
            display: none;
            justify-content: center;
            align-items: center;
            z-index: 9999;
            opacity: 0;
            transition: opacity 0.3s ease;
        }

        .loading-overlay.active {
            display: flex;
            opacity: 1;
        }

        .loading-content {
            background: var(--card-background);
            padding: 2.5rem;
            border-radius: var(--border-radius);
            text-align: center;
            box-shadow: 0 8px 32px rgba(0, 0, 0, 0.2);
            max-width: 600px;
            width: 90%;
            position: relative;
            overflow: hidden;
        }

        .loading-content::before {
            content: '';
            position: absolute;
            top: 0;
            left: 0;
            right: 0;
            height: 4px;
            background: linear-gradient(90deg, var(--primary-color), var(--secondary-color));
        }

        .loading-spinner {
            width: 80px;
            height: 80px;
            margin: 0 auto 2rem;
            border: 4px solid var(--border-color);
            border-top: 4px solid var(--primary-color);
            border-radius: 50%;
            animation: spin 1s linear infinite;
        }

        .loading-text h3 {
            color: var(--text-primary);
            margin-bottom: 1rem;
            font-size: 1.5rem;
            font-weight: 600;
        }

        .loading-status {
            color: var(--text-secondary);
            font-size: 1rem;
            margin-bottom: 2rem;
        }

        .processing-steps {
            margin-top: 2rem;
            text-align: left;
            max-height: 300px;
            overflow-y: auto;
            padding-right: 1rem;
        }

        .processing-step {
            display: flex;
            align-items: center;
            gap: 1rem;
            margin-bottom: 1rem;
            opacity: 0.5;
            transition: all 0.3s ease;
            font-size: 1rem;
            padding: 1rem;
            border-radius: 8px;
            background: var(--background-color);
            position: relative;
            overflow: hidden;
        }

        .processing-step::before {
            content: '';
            position: absolute;
            left: 0;
            top: 0;
            height: 100%;
            width: 4px;
            background: var(--primary-color);
            opacity: 0;
            transition: opacity 0.3s ease;
        }

        .processing-step.active {
            opacity: 1;
            background: var(--primary-color);
            color: white;
            transform: translateX(10px);
        }

        .processing-step.active::before {
            opacity: 1;
        }

        .processing-step.completed {
            opacity: 0.8;
            background: var(--card-background);
            border: 1px solid var(--primary-color);
        }

        .processing-step.completed::before {
            content: '✓';
            color: #10B981;
            font-weight: bold;
            margin-right: 8px;
            background: none;
        }

        .processing-step.error::before {
            content: '⚠';
            color: #EF4444;
            margin-right: 8px;
            background: none;
        }

        .progress-container {
            margin-top: 2rem;
            background: var(--background-color);
            border-radius: 999px;
            height: 6px;
            overflow: hidden;
        }

        .progress-bar {
            height: 100%;
            background: linear-gradient(90deg, var(--primary-color), var(--secondary-color));
            width: 0;
            transition: width 0.3s ease;
        }

        @keyframes spin {
            0% { transform: rotate(0deg); }
            100% { transform: rotate(360deg); }
        }

        /* Custom scrollbar for processing steps */
        .processing-steps::-webkit-scrollbar {
            width: 8px;
        }

        .processing-steps::-webkit-scrollbar-track {
            background: var(--background-color);
            border-radius: 4px;
        }

        .processing-steps::-webkit-scrollbar-thumb {
            background: var(--primary-color);
            border-radius: 4px;
        }

        .processing-steps::-webkit-scrollbar-thumb:hover {
            background: var(--primary-dark);
        }

        /* Restore other necessary styles */
        .preset-buttons {
            display: grid;
            grid-template-columns: repeat(auto-fit, minmax(120px, 1fr));
            gap: 8px;
            margin-bottom: var(--spacing);
        }

        .preset-btn {
            padding: 8px;
            border: 1px solid var(--border-color);
            border-radius: calc(var(--border-radius) / 2);
            background: var(--card-background);
            color: var(--text-primary);
            cursor: pointer;
            transition: all 0.2s ease;
        }

        .preset-btn:hover {
            border-color: var(--primary-color);
            background: var(--primary-color);
            color: white;
        }

        .preset-btn.active {
            background: var(--primary-color);
            color: white;
            border-color: var(--primary-color);
        }

        .face-upload {
            border: 2px dashed var(--border-color);
            border-radius: var(--border-radius);
            padding: var(--spacing);
            text-align: center;
            cursor: pointer;
            transition: all 0.2s ease;
        }

        .face-upload:hover {
            border-color: var(--primary-color);
        }

        .face-preview {
            width: 100px;
            height: 100px;
            border-radius: 50%;
            object-fit: cover;
            margin: 0 auto 12px;
            display: none;
        }

        .face-preview.active {
            display: block;
        }

        .generation-type {
            display: flex;
            gap: 12px;
            margin-bottom: var(--spacing);
        }

        .generation-type label {
            flex: 1;
            padding: 12px;
            border: 1px solid var(--border-color);
            border-radius: var(--border-radius);
            cursor: pointer;
            text-align: center;
            transition: all 0.2s ease;
        }

        .generation-type input[type="radio"] {
            display: none;
        }

        .generation-type input[type="radio"]:checked + label {
            background: var(--primary-color);
            color: white;
            border-color: var(--primary-color);
        }

        .custom-select {
            position: relative;
            width: 100%;
            margin-bottom: 1rem;
        }

        .select-selected {
            background-color: var(--card-background);
            padding: 12px;
            border: 1px solid var(--border-color);
            border-radius: calc(var(--border-radius) / 2);
            cursor: pointer;
            display: flex;
            align-items: center;
            gap: 8px;
        }

        .select-selected img {
            width: 40px;
            height: 40px;
            border-radius: 50%;
            object-fit: cover;
        }

        .select-items {
            position: absolute;
            background-color: var(--card-background);
            top: 100%;
            left: 0;
            right: 0;
            z-index: 99;
            border: 1px solid var(--border-color);
            border-radius: calc(var(--border-radius) / 2);
            margin-top: 4px;
            max-height: 300px;
            overflow-y: auto;
            box-shadow: 0 4px 6px var(--shadow-color);
        }

        .select-hide {
            display: none;
        }

        .select-item {
            padding: 12px;
            cursor: pointer;
            display: flex;
            align-items: center;
            gap: 8px;
            transition: background-color 0.2s ease;
        }

        .select-item:hover {
            background-color: var(--background-color);
        }

        .option-preview {
            width: 40px;
            height: 40px;
            border-radius: 50%;
            object-fit: cover;
        }

        .select-arrow-active {
            border-color: var(--primary-color);
        }

        .settings-grid {
            display: grid;
            grid-template-columns: repeat(auto-fit, minmax(200px, 1fr));
            gap: 1rem;
            margin-bottom: 1rem;
        }

        .setting-item {
            display: flex;
            flex-direction: column;
            gap: 0.5rem;
        }

        .setting-item label {
            font-size: 0.875rem;
            color: var(--text-secondary);
        }

        .setting-item input[type="number"] {
            width: 100%;
            padding: 0.5rem;
            border: 1px solid var(--border-color);
            border-radius: calc(var(--border-radius) / 2);
            background: var(--card-background);
            color: var(--text-primary);
        }

        .setting-item input[type="number"]:focus {
            outline: none;
            border-color: var(--primary-color);
        }

        /* Add these new styles */
        .step-icon {
            width: 40px;
            height: 40px;
            display: flex;
            align-items: center;
            justify-content: center;
            background: var(--background-color);
            border-radius: 50%;
            margin-right: 1rem;
            transition: all 0.3s ease;
        }

        .step-content {
            flex: 1;
        }

        .step-title {
            font-weight: 600;
            margin-bottom: 0.25rem;
        }

        .step-description {
            font-size: 0.875rem;
            color: var(--text-secondary);
            opacity: 0.8;
        }

        .processing-step {
            display: flex;
            align-items: center;
            padding: 1rem;
            margin-bottom: 1rem;
            background: var(--card-background);
            border-radius: 12px;
            border: 1px solid var(--border-color);
            transition: all 0.3s ease;
        }

        .processing-step.active {
            background: var(--primary-color);
            border-color: var(--primary-color);
            transform: translateX(10px);
        }

        .processing-step.active .step-icon {
            background: rgba(255, 255, 255, 0.2);
        }

        .processing-step.active .step-title,
        .processing-step.active .step-description {
            color: white;
        }

        .processing-step.completed {
            background: var(--card-background);
            border-color: var(--primary-color);
        }

        .processing-step.completed .step-icon {
            background: var(--primary-color);
            color: white;
        }

        .processing-step.completed .step-title {
            color: var(--primary-color);
        }

        .processing-step.error {
            background: #FEE2E2;
            border-color: #EF4444;
        }

        .processing-step.error .step-icon {
            background: #EF4444;
            color: white;
        }

        .processing-step.error .step-title {
            color: #EF4444;
        }

        @keyframes pulse {
            0% { transform: scale(1); }
            50% { transform: scale(1.05); }
            100% { transform: scale(1); }
        }

        .processing-step.active .step-icon {
            animation: pulse 1.5s infinite;
        }

        .modal-prompt {
            color: var(--text-primary);
            font-size: 1rem;
            line-height: 1.5;
            white-space: pre-wrap;
            padding: 15px;
            background: var(--background-color);
            border-radius: var(--border-radius);
            margin-top: 10px;
        }

        .modal-info-actions {
            margin-top: 15px;
            display: flex;
            flex-direction: column;
            align-items: stretch;
            gap: 12px;
        }

        .modal-info-actions .btn {
            width: 100%;
            justify-content: center;
            margin-top: 0;
        }

        /* Add styles for archive buttons */
        .nav__archive-buttons {
            display: flex;
            gap: 10px;
            margin-right: 15px;
        }

        /* Toast notification styles */
        .toast-container {
            position: fixed;
            top: 20px;
            right: 20px;
            z-index: 9999;
        }

        .toast {
            display: flex;
            align-items: flex-start;
            gap: 12px;
            background: var(--card-background);
            border-left: 4px solid var(--primary-color);
            padding: 16px;
            border-radius: 8px;
            box-shadow: 0 4px 12px rgba(0, 0, 0, 0.15);
            margin-bottom: 10px;
            max-width: 350px;
            transform: translateX(400px);
            opacity: 0;
            transition: all 0.3s ease;
        }

        .toast.show {
            transform: translateX(0);
            opacity: 1;
        }

        .toast.success {
            border-left-color: #10B981;
        }

        .toast.error {
            border-left-color: #EF4444;
        }

        .toast-icon {
            flex: 0 0 24px;
            height: 24px;
            display: flex;
            align-items: center;
            justify-content: center;
        }

        .toast-content {
            flex: 1;
        }

        .toast-title {
            font-weight: 600;
            margin-bottom: 4px;
            color: var(--text-primary);
        }

        .toast-message {
            color: var(--text-secondary);
            font-size: 0.875rem;
        }

        .toast-close {
            background: none;
            border: none;
            color: var(--text-secondary);
            cursor: pointer;
            padding: 0;
            font-size: 18px;
            line-height: 1;
        }

        /* Backup restore dialog */
        .backup-dialog {
            display: none;
            position: fixed;
            top: 0;
            left: 0;
            width: 100%;
            height: 100%;
            background: rgba(0, 0, 0, 0.5);
            z-index: 9999;
            justify-content: center;
            align-items: center;
        }

        .backup-dialog.show {
            display: flex;
        }

        .backup-dialog-content {
            background: var(--card-background);
            border-radius: var(--border-radius);
            padding: 24px;
            max-width: 500px;
            width: 90%;
        }

        .backup-dialog-title {
            font-size: 1.25rem;
            font-weight: 600;
            margin-bottom: 12px;
            color: var(--text-primary);
        }

        .backup-dialog-message {
            color: var(--text-secondary);
            margin-bottom: 20px;
        }

        .backup-dialog-actions {
            display: flex;
            justify-content: flex-end;
            gap: 12px;
        }
    </style>
</head>
<body>
    <div class="container">
        <nav class="nav">
            <div class="nav__title">AI Character Generator</div>
            <div class="nav__actions">
                <div class="nav__archive-buttons">
                    <a href="/save_to_drive" class="btn btn-secondary">
                        <svg width="20" height="20" viewBox="0 0 24 24" fill="none" xmlns="http://www.w3.org/2000/svg">
                            <path d="M19 9h-4V3H9v6H5l7 7 7-7zM5 18v2h14v-2H5z" fill="currentColor"/>
                        </svg>
                        Save to Drive
                    </a>
                    <a href="/restore_from_drive" class="btn btn-secondary">
                        <svg width="20" height="20" viewBox="0 0 24 24" fill="none" xmlns="http://www.w3.org/2000/svg">
                            <path d="M5 5v14h14V5H5zm12 12H7V7h10v10z" fill="currentColor"/>
                            <path d="M10 14l5-3-5-3v6z" fill="currentColor"/>
                        </svg>
                        Restore from Drive
                    </a>
                </div>
                <button class="theme-toggle" id="themeToggle" aria-label="Toggle theme">
                    <svg width="24" height="24" viewBox="0 0 24 24" fill="none" xmlns="http://www.w3.org/2000/svg">
                        <path d="M12 3C7.02944 3 3 7.02944 3 12C3 16.9706 7.02944 21 12 21C16.9706 21 21 16.9706 21 12C21 7.02944 16.9706 3 12 3ZM12 19C8.13401 19 5 15.866 5 12C5 8.13401 8.13401 5 12 5C15.866 5 19 8.13401 19 12C19 15.866 15.866 19 12 19Z" fill="currentColor"/>
                    </svg>
                </button>
                <div class="nav__downloads">
                    <a href="/download_pdf" class="btn btn-primary">
                        <svg width="20" height="20" viewBox="0 0 20 20" fill="none" xmlns="http://www.w3.org/2000/svg">
                            <path d="M10 12.5L6.5 9L7.5 8L9.5 10V3H10.5V10L12.5 8L13.5 9L10 12.5Z" fill="currentColor"/>
                            <path d="M3 15V16H17V15H3Z" fill="currentColor"/>
                        </svg>
                        Download PDF Report
                    </a>
                    <a href="/download_zip" class="btn btn-secondary">
                        <svg width="20" height="20" viewBox="0 0 20 20" fill="none" xmlns="http://www.w3.org/2000/svg">
                            <path d="M10 12.5L6.5 9L7.5 8L9.5 10V3H10.5V10L12.5 8L13.5 9L10 12.5Z" fill="currentColor"/>
                            <path d="M3 15V16H17V15H3Z" fill="currentColor"/>
                        </svg>
                        Download All Images
                    </a>
                </div>
            </div>
        </nav>

        <div class="main-content">
            <div class="control-panel">
                <form method="POST" action="{{ url_for('run_generation') }}" enctype="multipart/form-data" id="generationForm">
                    <div class="form-group">
                        <label>Generation Type</label>
                        <div class="generation-type">
                            <input type="radio" name="gen_type" id="gen_face" value="face" checked>
                            <label for="gen_face">Generate with Face</label>
                            <input type="radio" name="gen_type" id="gen_fashion" value="fashion">
                            <label for="gen_fashion">Fashion Design</label>
                            <input type="radio" name="gen_type" id="gen_prompt" value="prompt">
                            <label for="gen_prompt">Image to Prompt</label>
                        </div>
                    </div>

                    <!-- Face Generation Section -->
                    <div id="face-gen-section">
                        <!-- Reface Option -->
                        <div class="form-group">
                            <label class="checkbox-label">
                                <input type="checkbox" name="reface_mode" id="refaceMode">
                                <span class="checkbox-text">Reface your image (direct face swap)</span>
                            </label>
                            <p class="help-text">Use this option to apply your face to an existing image without AI generation</p>
                        </div>

                        <div class="form-group face-required" id="face-upload-section">
                            <label>Upload Face Image</label>
                            <div class="face-upload" id="faceUpload">
                                <img class="face-preview" id="facePreview" src="" alt="Face Preview">
                                <p>Click to upload or drag and drop</p>
                                <input type="file" name="face_upload" id="faceInput" accept="image/*" style="display: none">
                            </div>
                        </div>

                        <div class="form-group" id="face-select-section">
                            <label>Select Previously Used Face</label>
                            <div class="custom-select">
                                <div class="select-selected">-- none --</div>
                                <div class="select-items select-hide">
                                    <div class="select-item" data-value="">-- none --</div>
                                    {% for f in faces %}
                                    <div class="select-item" data-value="{{ f }}">
                                        <img src="{{ url_for('serve_face', filename=f) }}" alt="{{ f }}" class="option-preview">
                                        <span>{{ f }}</span>
                                    </div>
                                    {% endfor %}
                                </div>
                                <input type="hidden" name="face_select" value="">
                            </div>
                        </div>

                        <!-- Reface Target Image Upload -->
                        <div class="form-group" id="reface-target-section" style="display: none;">
                            <label class="required">Upload Target Image (Image to Reface)</label>
                            <div class="face-upload" id="refaceTargetUpload">
                                <img class="face-preview" id="refaceTargetPreview" src="" alt="Target Image Preview">
                                <p>Click to upload or drag and drop the image you want to apply your face to</p>
                                <input type="file" name="reface_target_upload" id="refaceTargetInput" accept="image/*" style="display: none">
                            </div>
                        </div>

                        <!-- Standard Generation Fields (hidden in reface mode) -->
                        <div id="standard-generation-fields">
                            <div class="form-group">
                                <label class="required">Text Prompt</label>
                                <textarea name="text_prompt" class="form-control" rows="3"
                                        placeholder="Describe your character..." required></textarea>
                            </div>

                            <div class="form-group">
                                <label>Body Type</label>
                                <div class="preset-buttons">
                                    <button type="button" class="preset-btn" data-type="slim">Slim</button>
                                    <button type="button" class="preset-btn" data-type="athletic">Athletic</button>
                                    <button type="button" class="preset-btn" data-type="curvy">Curvy</button>
                                    <button type="button" class="preset-btn" data-type="chubby">Chubby</button>
                                    <button type="button" class="preset-btn" data-type="plus-size">Plus Size</button>
                                </div>
                                <input type="hidden" name="body_type" id="bodyType" value="none">
                            </div>

                            <div class="form-group">
                                <label>Breast Size</label>
                                <select name="breast_size" class="form-control">
                                    <option value="none">none</option>
                                    <option value="small breasts">Small</option>
                                    <option value="medium breasts">Medium</option>
                                    <option value="large breasts">Large</option>
                                    <option value="very large breasts">Very Large</option>
                                    <option value="flat chest">Flat</option>
                                </select>
                            </div>

                            <div class="form-group">
                                <label>Hip Shape</label>
                                <select name="hip_shape" class="form-control">
                                    <option value="none">none</option>
                                    <option value="narrow hips">Narrow</option>
                                    <option value="average hips">Average</option>
                                    <option value="wide hips">Wide</option>
                                    <option value="very wide hips">Very Wide</option>
                                </select>
                            </div>

                            <div class="form-group">
                                <label>Other Traits</label>
                                <input type="text" name="other_traits" class="form-control"
                                       placeholder="e.g. soft belly, thick thighs">
                            </div>
                        </div>
                    </div>

                    <!-- Fashion Design Section -->
                    <div id="fashion-design-section" style="display: none;">
                        <div class="form-group">
                            <label>Upload Base Image</label>
                            <div class="face-upload" id="fashionUpload">
                                <img class="face-preview" id="fashionPreview" src="" alt="Fashion Preview">
                                <p>Click to upload or drag and drop</p>
                                <input type="file" name="fashion_upload" id="fashionInput" accept="image/*" style="display: none">
                            </div>
                        </div>

                        <div class="form-group">
                            <label>Design Prompt</label>
                            <textarea name="fashion_prompt" class="form-control" rows="3"
                                    placeholder="Describe the fashion design you want..."></textarea>
                        </div>

                        <div class="form-group">
                            <label class="checkbox-label">
                                <input type="checkbox" name="restore_face" id="restoreFace">
                                Restore face quality after design
                            </label>
                            <p class="help-text">This will apply face swap to restore the original face quality after fashion design</p>
                        </div>
                    </div>

                    <!-- Image to Prompt Section -->
                    <div id="prompt-section" style="display: none;">
                        <div class="form-group">
                            <label>Upload Image</label>
                            <div class="face-upload" id="promptUpload">
                                <img class="face-preview" id="promptPreview" src="" alt="Image Preview">
                                <p>Click to upload or drag and drop</p>
                                <input type="file" name="prompt_upload" id="promptInput" accept="image/*" style="display: none">
                            </div>
                        </div>

                        <div class="form-group">
                            <label>Generation Settings</label>
                            <div class="settings-grid">
                                <div class="setting-item">
                                    <label for="top_k">Top K Style Keywords</label>
                                    <input type="number" name="top_k" id="top_k" class="form-control" value="5" min="1" max="15">
                                </div>
                                <div class="setting-item">
                                    <label for="max_length">Max Length</label>
                                    <input type="number" name="max_length" id="max_length" class="form-control" value="50" min="20" max="100">
                                </div>
                                <div class="setting-item">
                                    <label for="min_length">Min Length</label>
                                    <input type="number" name="min_length" id="min_length" class="form-control" value="20" min="10" max="50">
                                </div>
                                <div class="setting-item">
                                    <label for="num_beams">Number of Beams</label>
                                    <input type="number" name="num_beams" id="num_beams" class="form-control" value="5" min="1" max="10">
                                </div>
                            </div>
                        </div>

                        <div class="form-group">
                            <label>Generated Prompt</label>
                            <textarea name="generated_prompt" id="generatedPrompt" class="form-control" rows="3" readonly></textarea>
                            <button type="button" class="btn btn-secondary" id="copyPromptBtn" style="margin-top: 10px;">
                                <svg width="20" height="20" viewBox="0 0 20 20" fill="none" xmlns="http://www.w3.org/2000/svg">
                                    <path d="M8 2C7.44772 2 7 2.44772 7 3V4H5C3.89543 4 3 4.89543 3 6V16C3 17.1046 3.89543 18 5 18H13C14.1046 18 15 17.1046 15 16V14H16C16.5523 14 17 13.5523 17 13V5C17 3.89543 16.1046 3 15 3H13V2C13 1.44772 12.5523 1 12 1H8C7.44772 1 7 1.44772 7 2ZM13 4H15V13H13V4ZM5 6H13V16H5V6Z" fill="currentColor"/>
                                </svg>
                                Copy Prompt
                            </button>
                        </div>
                    </div>

                    <button type="submit" class="btn btn-primary" style="width: 100%">
                        Generate
                    </button>
                </form>
            </div>

            <div class="gallery">
                <h2>Generation History</h2>
                <div class="gallery-grid">
                    {% for base, final, prompt, is_fashion, is_swapped in history %}
                    <div class="gallery-item" data-images='[
                        {% if final %}
                        {"src": "{{ url_for('serve_swapped', filename=final) }}", "title": "{% if is_fashion %}Fashion Design (Face Swapped){% else %}Face Swapped{% endif %}"},
                        {"src": "{{ url_for('serve_generated', filename=base) }}", "title": "{% if is_fashion %}Fashion Design{% else %}Generated Image{% endif %}"}
                        {% else %}
                        {"src": "{{ url_for('serve_generated', filename=base) }}", "title": "{% if is_fashion %}Fashion Design{% else %}Generated Image{% endif %}"}
                        {% endif %}
                    ]' data-prompt="{{ prompt }}">
                        <img src="{% if final %}{{ url_for('serve_swapped', filename=final) }}{% else %}{{ url_for('serve_generated', filename=base) }}{% endif %}" alt="Generated">
                        <div class="gallery-item__overlay">
                            <div class="gallery-item__title">
                                {% if is_fashion %}
                                Fashion Design
                                {% else %}
                                Generated Image
                                {% endif %}
                                {% if final %}
                                (Face Swapped)
                                {% endif %}
                            </div>
                            <div class="gallery-item__prompt">{{ prompt }}</div>
                            <div class="gallery-item__actions">
                                <button class="gallery-item__download" data-src="{% if final %}{{ url_for('serve_swapped', filename=final) }}{% else %}{{ url_for('serve_generated', filename=base) }}{% endif %}">
                                    <svg width="16" height="16" viewBox="0 0 20 20" fill="none" xmlns="http://www.w3.org/2000/svg">
                                        <path d="M10 12.5L6.5 9L7.5 8L9.5 10V3H10.5V10L12.5 8L13.5 9L10 12.5Z" fill="currentColor"/>
                                        <path d="M3 15V16H17V15H3Z" fill="currentColor"/>
                                    </svg>
                                    Download
                                </button>
                                <button class="gallery-item__delete" data-base="{% if base %}{{ base }}{% endif %}" data-final="{% if final %}{{ final }}{% endif %}">
                                    <svg width="16" height="16" viewBox="0 0 20 20" fill="none" xmlns="http://www.w3.org/2000/svg">
                                        <path d="M8.6 14L10 12.6L11.4 14L12.4 13L11 11.6L12.4 10.2L11.4 9.2L10 10.6L8.6 9.2L7.6 10.2L9 11.6L7.6 13L8.6 14ZM6 18C5.45 18 4.98 17.8 4.58 17.4C4.18 17 3.98 16.55 3.98 16L4 6H3V4H7V3H13V4H17V6H16V16C16 16.55 15.8 17 15.4 17.4C15 17.8 14.55 18 14 18H6Z" fill="currentColor"/>
                                    </svg>
                                    Delete
                                </button>
                            </div>
                        </div>
                    </div>
                    {% else %}
                    <p class="text-muted">No generations yet.</p>
                    {% endfor %}
                </div>
            </div>
        </div>
    </div>

    <!-- Image Modal -->
    <div class="modal" id="imageModal">
        <div class="modal-content">
            <button class="modal-close">&times;</button>
            <div class="modal-layout">
                <div class="modal-image-container">
                    <img class="modal-image" id="modalImage" src="" alt="Enlarged view">
                    <div class="modal-actions">
                        <!-- Removed fashion design button from here -->
                    </div>
                    <div class="modal-nav">
                        <button class="modal-nav-btn" id="prevBtn" disabled>&lt;</button>
                        <button class="modal-nav-btn" id="nextBtn" disabled>&gt;</button>
                    </div>
                    <div class="modal-counter" id="imageCounter"></div>
                </div>
                <div class="modal-info">
                    <div class="modal-prompt" id="modalPrompt"></div>
                    <div class="modal-info-actions">
                        <button class="btn btn-secondary download-image-btn" id="downloadImageBtn">
                            <svg width="20" height="20" viewBox="0 0 20 20" fill="none" xmlns="http://www.w3.org/2000/svg">
                                <path d="M10 12.5L6.5 9L7.5 8L9.5 10V3H10.5V10L12.5 8L13.5 9L10 12.5Z" fill="currentColor"/>
                                <path d="M3 15V16H17V15H3Z" fill="currentColor"/>
                            </svg>
                            Download Image
                        </button>
                        <button class="btn btn-primary fashion-design-btn" id="fashionDesignBtn" style="margin-top: 10px;">
                            <svg width="20" height="20" viewBox="0 0 20 20" fill="none" xmlns="http://www.w3.org/2000/svg">
                                <path d="M10 2C5.58172 2 2 5.58172 2 10C2 14.4183 5.58172 18 10 18C14.4183 18 18 14.4183 18 10C18 5.58172 14.4183 2 10 2ZM10 16C6.68629 16 4 13.3137 4 10C4 6.68629 6.68629 4 10 4C13.3137 4 16 6.68629 16 10C16 13.3137 13.3137 16 10 16Z" fill="currentColor"/>
                            </svg>
                            Fashion Design
                        </button>
                    </div>
                </div>
            </div>
        </div>
    </div>

    <!-- Loading Overlay -->
    <div class="loading-overlay" id="loadingOverlay">
        <div class="loading-content">
            <div class="loading-spinner"></div>
            <div class="loading-text">
                <h3 id="loadingTitle">Processing</h3>
                <p class="loading-status" id="loadingStatus">Initializing...</p>
            </div>

            <!-- Face Generation Steps -->
            <div class="processing-steps" id="faceGenerationSteps" style="display: none;">
                <div class="processing-step" data-step="init">Initializing AI model...</div>
                <div class="processing-step" data-step="base">Generating base image...</div>
                <div class="processing-step" data-step="face">Processing face details...</div>
                <div class="processing-step" data-step="swap">Applying face swap...</div>
                <div class="processing-step" data-step="enhance">Enhancing final result...</div>
                <div class="processing-step" data-step="optimize">Optimizing image quality...</div>
                <div class="processing-step" data-step="finalize">Finalizing details...</div>
            </div>

            <!-- Fashion Design Steps -->
            <div class="processing-steps" id="fashionDesignSteps" style="display: none;">
                <div class="processing-step" data-step="init">Initializing fashion design model...</div>
                <div class="processing-step" data-step="base">Processing base image...</div>
                <div class="processing-step" data-step="mask">Generating design mask...</div>
                <div class="processing-step" data-step="design">Creating fashion design...</div>
                <div class="processing-step" data-step="enhance">Enhancing design details...</div>
                <div class="processing-step" data-step="face">Restoring face quality...</div>
                <div class="processing-step" data-step="finalize">Finalizing design...</div>
        </div>

            <!-- Generate Only Steps -->
            <div class="processing-steps" id="generateOnlySteps" style="display: none;">
                <div class="processing-step" data-step="init">Initializing AI model...</div>
                <div class="processing-step" data-step="base">Generating base image...</div>
                <div class="processing-step" data-step="enhance">Enhancing image quality...</div>
                <div class="processing-step" data-step="optimize">Optimizing details...</div>
                <div class="processing-step" data-step="finalize">Finalizing result...</div>
            </div>

            <!-- Image to Prompt Steps -->
            <div class="processing-steps" id="promptGenerationSteps" style="display: none;">
                <div class="processing-step" data-step="init">
                    <div class="step-icon">
                        <svg width="24" height="24" viewBox="0 0 24 24" fill="none" xmlns="http://www.w3.org/2000/svg">
                            <path d="M12 2C6.48 2 2 6.48 2 12C2 17.52 6.48 22 12 22C17.52 22 22 17.52 22 12C22 6.48 17.52 2 12 2ZM12 20C7.59 20 4 16.41 4 12C4 7.59 7.59 4 12 4C16.41 4 20 7.59 20 12C20 16.41 16.41 20 12 20Z" fill="currentColor"/>
                            <path d="M12 6C8.69 6 6 8.69 6 12C6 15.31 8.69 18 12 18C15.31 18 18 15.31 18 12C18 8.69 15.31 6 12 6ZM12 16C9.79 16 8 14.21 8 12C8 9.79 9.79 8 12 8C14.21 8 16 9.79 16 12C16 14.21 14.21 16 12 16Z" fill="currentColor"/>
                        </svg>
                    </div>
                    <div class="step-content">
                        <div class="step-title">Initializing prompt models...</div>
                        <div class="step-description">Loading BLIP and CLIP models for analysis</div>
                    </div>
                </div>
                <div class="processing-step" data-step="blip">
                    <div class="step-icon">
                        <svg width="24" height="24" viewBox="0 0 24 24" fill="none" xmlns="http://www.w3.org/2000/svg">
                            <path d="M21 19V5C21 3.9 20.1 3 19 3H5C3.9 3 3 3.9 3 5V19C3 20.1 3.9 21 5 21H19C20.1 21 21 20.1 21 19ZM8.5 13.5L11 16.51L14.5 12L19 18H5L8.5 13.5Z" fill="currentColor"/>
                        </svg>
                    </div>
                    <div class="step-content">
                        <div class="step-title">Generating image caption...</div>
                        <div class="step-description">Analyzing image content and generating descriptive text</div>
                    </div>
                </div>
                <div class="processing-step" data-step="clip">
                    <div class="step-icon">
                        <svg width="24" height="24" viewBox="0 0 24 24" fill="none" xmlns="http://www.w3.org/2000/svg">
                            <path d="M12 2C6.48 2 2 6.48 2 12C2 17.52 6.48 22 12 22C17.52 22 22 17.52 22 12C22 6.48 17.52 2 12 2ZM12 20C7.59 20 4 16.41 4 12C4 7.59 7.59 4 12 4C16.41 4 20 7.59 20 12C20 16.41 16.41 20 12 20Z" fill="currentColor"/>
                            <path d="M12 6C8.69 6 6 8.69 6 12C6 15.31 8.69 18 12 18C15.31 18 18 15.31 18 12C18 8.69 15.31 6 12 6ZM12 16C9.79 16 8 14.21 8 12C8 9.79 9.79 8 12 8C14.21 8 16 9.79 16 12C16 14.21 14.21 16 12 16Z" fill="currentColor"/>
                        </svg>
                    </div>
                    <div class="step-content">
                        <div class="step-title">Analyzing image style...</div>
                        <div class="step-description">Extracting style keywords and visual elements</div>
                    </div>
                </div>
                <div class="processing-step" data-step="combine">
                    <div class="step-icon">
                        <svg width="24" height="24" viewBox="0 0 24 24" fill="none" xmlns="http://www.w3.org/2000/svg">
                            <path d="M19 3H5C3.9 3 3 3.9 3 5V19C3 20.1 3.9 21 5 21H19C20.1 21 21 20.1 21 19V5C21 3.9 20.1 3 19 3ZM9 17H7V10H9V17ZM13 17H11V7H13V17ZM17 17H15V13H17V17Z" fill="currentColor"/>
                        </svg>
                    </div>
                    <div class="step-content">
                        <div class="step-title">Combining style keywords...</div>
                        <div class="step-description">Merging caption and style analysis into a cohesive prompt</div>
                    </div>
                </div>
                <div class="processing-step" data-step="finalize">
                    <div class="step-icon">
                        <svg width="24" height="24" viewBox="0 0 24 24" fill="none" xmlns="http://www.w3.org/2000/svg">
                            <path d="M9 16.17L4.83 12L3.41 13.41L9 19L21 7L19.59 5.59L9 16.17Z" fill="currentColor"/>
                        </svg>
                    </div>
                    <div class="step-content">
                        <div class="step-title">Finalizing prompt...</div>
                        <div class="step-description">Optimizing and formatting the final prompt</div>
                    </div>
                </div>
            </div>

            <style>
                /* Add these new styles */
                .step-icon {
                    width: 40px;
                    height: 40px;
                    display: flex;
                    align-items: center;
                    justify-content: center;
                    background: var(--background-color);
                    border-radius: 50%;
                    margin-right: 1rem;
                    transition: all 0.3s ease;
                }

                .step-content {
                    flex: 1;
                }

                .step-title {
                    font-weight: 600;
                    margin-bottom: 0.25rem;
                }

                .step-description {
                    font-size: 0.875rem;
                    color: var(--text-secondary);
                    opacity: 0.8;
                }

                .processing-step {
                    display: flex;
                    align-items: center;
                    padding: 1rem;
                    margin-bottom: 1rem;
                    background: var(--card-background);
                    border-radius: 12px;
                    border: 1px solid var(--border-color);
                    transition: all 0.3s ease;
                }

                .processing-step.active {
                    background: var(--primary-color);
                    border-color: var(--primary-color);
                    transform: translateX(10px);
                }

                .processing-step.active .step-icon {
                    background: rgba(255, 255, 255, 0.2);
                }

                .processing-step.active .step-title,
                .processing-step.active .step-description {
                    color: white;
                }

                .processing-step.completed {
                    background: var(--card-background);
                    border-color: var(--primary-color);
                }

                .processing-step.completed .step-icon {
                    background: var(--primary-color);
                    color: white;
                }

                .processing-step.completed .step-title {
                    color: var(--primary-color);
                }

                .processing-step.error {
                    background: #FEE2E2;
                    border-color: #EF4444;
                }

                .processing-step.error .step-icon {
                    background: #EF4444;
                    color: white;
                }

                .processing-step.error .step-title {
                    color: #EF4444;
                }

                @keyframes pulse {
                    0% { transform: scale(1); }
                    50% { transform: scale(1.05); }
                    100% { transform: scale(1); }
                }

                .processing-step.active .step-icon {
                    animation: pulse 1.5s infinite;
                }
            </style>

            <script>
                // Update the prompt generation handling
                if (genType === 'prompt') {
                    const promptInput = document.getElementById('promptInput');

                    if (!promptInput.files.length) {
                        alert('Please upload an image for prompt generation');
                        return;
                    }

                    loadingOverlay.classList.add('active');
                    loadingStatus.textContent = 'Initializing prompt generation...';

                    const promptFormData = new FormData();
                    promptFormData.append('prompt_image', promptInput.files[0]);
                    promptFormData.append('top_k', document.getElementById('top_k').value);
                    promptFormData.append('max_length', document.getElementById('max_length').value);
                    promptFormData.append('min_length', document.getElementById('min_length').value);
                    promptFormData.append('num_beams', document.getElementById('num_beams').value);

                    // Update progress bar for prompt generation
                    const steps = processingSteps.querySelectorAll('.processing-step');
                    let currentStep = 0;

                    const updateProgress = () => {
                        const progress = ((currentStep + 1) / steps.length) * 100;
                        progressBar.style.width = `${progress}%`;

                        // Update step status
                        steps.forEach((step, index) => {
                            step.classList.remove('active', 'completed', 'error');
                            if (index < currentStep) {
                                step.classList.add('completed');
                            } else if (index === currentStep) {
                                step.classList.add('active');
                            }
                        });
                    };

                    // Simulate step progression
                    const stepDurations = {
                        'init': 2000,
                        'blip': 3000,
                        'clip': 2500,
                        'combine': 2000,
                        'finalize': 1500
                    };

                    const simulateSteps = () => {
                        const stepNames = ['init', 'blip', 'clip', 'combine', 'finalize'];
                        let currentIndex = 0;

                        const processStep = () => {
                            if (currentIndex < stepNames.length) {
                                const stepName = stepNames[currentIndex];
                                loadingStatus.textContent = steps[currentIndex].querySelector('.step-title').textContent;
                                currentStep = currentIndex;
                                updateProgress();

                                setTimeout(() => {
                                    currentIndex++;
                                    processStep();
                                }, stepDurations[stepName]);
                            }
                        };

                        processStep();
                    };

                    simulateSteps();

                    fetch('/generate_prompt', {
                        method: 'POST',
                        body: promptFormData
                    })
                    .then(response => response.json())
                    .then(data => {
                        loadingOverlay.classList.remove('active');
                        if (data.success) {
                            document.getElementById('generatedPrompt').value = data.prompt;
                        } else {
                            alert('Error generating prompt: ' + data.message);
                        }
                    })
                    .catch(error => {
                        loadingOverlay.classList.remove('active');
                        console.error('Error generating prompt:', error);
                        alert('Error generating prompt');
                    });

                    return;
                }
            </script>

            <!-- Progress Bar -->
            <div class="progress-container">
                <div class="progress-bar" id="progressBar"></div>
            </div>
        </div>
    </div>

    <script>
        // Theme toggle
        const themeToggle = document.getElementById('themeToggle');
        const prefersDarkScheme = window.matchMedia('(prefers-color-scheme: dark)');

        const currentTheme = localStorage.getItem('theme') ||
            (prefersDarkScheme.matches ? 'dark' : 'light');

        document.body.setAttribute('data-theme', currentTheme);

        themeToggle.addEventListener('click', () => {
            const newTheme = document.body.getAttribute('data-theme') === 'dark' ? 'light' : 'dark';
            document.body.setAttribute('data-theme', newTheme);
            localStorage.setItem('theme', newTheme);
        });

        // Face upload preview
        const faceUpload = document.getElementById('faceUpload');
        const faceInput = document.getElementById('faceInput');
        const facePreview = document.getElementById('facePreview');

        faceUpload.addEventListener('click', () => faceInput.click());
        faceUpload.addEventListener('dragover', (e) => {
            e.preventDefault();
            faceUpload.style.borderColor = 'var(--primary-color)';
        });
        faceUpload.addEventListener('dragleave', () => {
            faceUpload.style.borderColor = 'var(--border-color)';
        });
        faceUpload.addEventListener('drop', (e) => {
            e.preventDefault();
            faceUpload.style.borderColor = 'var(--border-color)';
            const file = e.dataTransfer.files[0];
            if (file && file.type.startsWith('image/')) {
                handleFile(file);
            }
        });

        faceInput.addEventListener('change', (e) => {
            const file = e.target.files[0];
            if (file) {
                handleFile(file);
            }
        });

        function handleFile(file) {
            const reader = new FileReader();
            reader.onload = (e) => {
                facePreview.src = e.target.result;
                facePreview.classList.add('active');
            };
            reader.readAsDataURL(file);
        }

        // Body type presets
        const presetBtns = document.querySelectorAll('.preset-btn');
        const bodyTypeInput = document.getElementById('bodyType');

        presetBtns.forEach(btn => {
            btn.addEventListener('click', () => {
                presetBtns.forEach(b => b.classList.remove('active'));
                btn.classList.add('active');
                bodyTypeInput.value = btn.dataset.type;
            });
        });

        // Generation type toggle
        const genTypeInputs = document.querySelectorAll('input[name="gen_type"]');
        const faceGenSection = document.getElementById('face-gen-section');
        const fashionDesignSection = document.getElementById('fashion-design-section');
        const promptSection = document.getElementById('prompt-section');
        const faceUploadSection = document.getElementById('face-upload-section');
        const faceSelectSection = document.getElementById('face-select-section');
        const refaceTargetSection = document.getElementById('reface-target-section');
        const standardGenerationFields = document.getElementById('standard-generation-fields');
        const refaceModeCheckbox = document.getElementById('refaceMode');
        const generationForm = document.getElementById('generationForm');

        // Handle reface mode toggle
        refaceModeCheckbox.addEventListener('change', function() {
            const faceGenSection = document.querySelector('#face-gen-section');
            const textPromptElement = document.querySelector('textarea[name="text_prompt"]');

            if (this.checked) {
                faceGenSection.classList.add('reface-active');
                // Set reface target as required
                document.querySelector('#refaceTargetInput').setAttribute('required', '');
                // Make at least one face source required
                document.querySelector('#faceInput').setAttribute('required', '');
                // Text prompt no longer required
                if (textPromptElement) {
                    textPromptElement.removeAttribute('required');
                    // Also clear required validation message
                    textPromptElement.setCustomValidity('');
                }
            } else {
                faceGenSection.classList.remove('reface-active');
                // Reset requirements
                document.querySelector('#refaceTargetInput').removeAttribute('required');
                document.querySelector('#faceInput').removeAttribute('required');
                if (textPromptElement) {
                    textPromptElement.setAttribute('required', '');
                }
            }
        });

        // Handle reface target upload
        const refaceTargetUpload = document.getElementById('refaceTargetUpload');
        const refaceTargetInput = document.getElementById('refaceTargetInput');
        const refaceTargetPreview = document.getElementById('refaceTargetPreview');

        if (refaceTargetUpload && refaceTargetInput) {
            refaceTargetUpload.addEventListener('click', () => {
                refaceTargetInput.click();
            });

            refaceTargetUpload.addEventListener('dragover', (e) => {
                e.preventDefault();
                refaceTargetUpload.style.borderColor = 'var(--primary-color)';
            });

            refaceTargetUpload.addEventListener('dragleave', () => {
                refaceTargetUpload.style.borderColor = 'var(--border-color)';
            });

            refaceTargetUpload.addEventListener('drop', (e) => {
                e.preventDefault();
                refaceTargetUpload.style.borderColor = 'var(--border-color)';
                const file = e.dataTransfer.files[0];
                if (file && file.type.startsWith('image/')) {
                    handleRefaceTargetFile(file);
                }
            });

            refaceTargetInput.addEventListener('change', (e) => {
                const file = e.target.files[0];
                if (file) {
                    handleRefaceTargetFile(file);
                }
            });
        }

        function handleRefaceTargetFile(file) {
            const reader = new FileReader();
            reader.onload = (e) => {
                refaceTargetPreview.src = e.target.result;
                refaceTargetPreview.classList.add('active');
            };
            reader.readAsDataURL(file);
        }

        genTypeInputs.forEach(input => {
            input.addEventListener('change', () => {
                if (input.value === 'fashion') {
                    faceGenSection.style.display = 'none';
                    fashionDesignSection.style.display = 'block';
                    promptSection.style.display = 'none';
                    // Clear face generation inputs
                    document.querySelectorAll('#face-gen-section input, #face-gen-section textarea').forEach(el => {
                        el.removeAttribute('required');
                    });
                    // Set fashion inputs as required
                    document.querySelector('#fashionInput').setAttribute('required', '');
                    document.querySelector('textarea[name="fashion_prompt"]').setAttribute('required', '');
                } else if (input.value === 'face') {
                    faceGenSection.style.display = 'block';
                    fashionDesignSection.style.display = 'none';
                    promptSection.style.display = 'none';
                    faceUploadSection.style.display = 'block';
                    faceSelectSection.style.display = 'block';
                    // Clear fashion inputs
                    document.querySelector('#fashionInput').removeAttribute('required');
                    document.querySelector('textarea[name="fashion_prompt"]').removeAttribute('required');
                    // Set text prompt as required only if not in reface mode
                    if (!refaceModeCheckbox.checked) {
                        document.querySelector('textarea[name="text_prompt"]').setAttribute('required', '');
                    }
                } else if (input.value === 'prompt') {
                    faceGenSection.style.display = 'none';
                    fashionDesignSection.style.display = 'none';
                    promptSection.style.display = 'block';
                    faceUploadSection.style.display = 'none';
                    faceSelectSection.style.display = 'none';
                    // Clear other inputs
                    document.querySelector('#fashionInput').removeAttribute('required');
                    document.querySelector('textarea[name="fashion_prompt"]').removeAttribute('required');
                    document.querySelector('textarea[name="text_prompt"]').removeAttribute('required');
                    // Set prompt input as required
                    document.querySelector('#promptInput').setAttribute('required', '');
                }
            });
        });

        // Custom select functionality
        document.addEventListener('DOMContentLoaded', function() {
            const customSelect = document.querySelector('.custom-select');
            const selectSelected = customSelect.querySelector('.select-selected');
            const selectItems = customSelect.querySelector('.select-items');
            const hiddenInput = customSelect.querySelector('input[type="hidden"]');

            // Click on the select box
            selectSelected.addEventListener('click', function(e) {
                e.stopPropagation();
                selectItems.classList.toggle('select-hide');
                this.classList.toggle('select-arrow-active');
            });

            // Click on an option
            selectItems.querySelectorAll('.select-item').forEach(item => {
                item.addEventListener('click', function() {
                    const value = this.getAttribute('data-value');
                    const text = this.querySelector('span')?.textContent || this.textContent;
                    const img = this.querySelector('img');

                    // Update selected text and value
                    selectSelected.innerHTML = img ?
                        `<img src="${img.src}" alt="${text}">${text}` :
                        text;
                    hiddenInput.value = value;

                    // Hide the dropdown
                    selectItems.classList.add('select-hide');
                    selectSelected.classList.remove('select-arrow-active');
                });
            });

            // Close the dropdown when clicking outside
            document.addEventListener('click', function() {
                selectItems.classList.add('select-hide');
                selectSelected.classList.remove('select-arrow-active');
            });
        });

        // Gallery download buttons functionality
        document.querySelectorAll('.gallery-item__download').forEach(button => {
            button.addEventListener('click', function(e) {
                e.stopPropagation(); // Prevent opening the modal when clicking the download button

                const imageUrl = this.getAttribute('data-src');
                if (!imageUrl) return;

                // Create a temporary link element
                const downloadLink = document.createElement('a');
                downloadLink.href = imageUrl;

                // Extract filename from the URL
                const filename = imageUrl.split('/').pop();
                downloadLink.download = filename;

                // Append to body, click and remove
                document.body.appendChild(downloadLink);
                downloadLink.click();
                document.body.removeChild(downloadLink);

                // Show feedback
                const originalText = this.innerHTML;
                this.innerHTML = '<svg width="16" height="16" viewBox="0 0 20 20" fill="none" xmlns="http://www.w3.org/2000/svg"><path d="M8 15L3 10L4.41 8.59L8 12.17L15.59 4.58L17 6L8 15Z" fill="currentColor"/></svg>Downloaded';
                setTimeout(() => {
                    this.innerHTML = originalText;
                }, 2000);
            });
        });

        // Fashion upload handling
        const fashionUpload = document.getElementById('fashionUpload');
        const fashionInput = document.getElementById('fashionInput');
        const fashionPreview = document.getElementById('fashionPreview');

        if (fashionUpload && fashionInput) {
            fashionUpload.addEventListener('click', () => {
                fashionInput.click();
            });

            fashionUpload.addEventListener('dragover', (e) => {
                e.preventDefault();
                fashionUpload.style.borderColor = 'var(--primary-color)';
            });

            fashionUpload.addEventListener('dragleave', () => {
                fashionUpload.style.borderColor = 'var(--border-color)';
            });

            fashionUpload.addEventListener('drop', (e) => {
                e.preventDefault();
                fashionUpload.style.borderColor = 'var(--border-color)';
                const file = e.dataTransfer.files[0];
                if (file && file.type.startsWith('image/')) {
                    handleFashionFile(file);
                }
            });

            fashionInput.addEventListener('change', (e) => {
                const file = e.target.files[0];
                if (file) {
                    handleFashionFile(file);
                }
            });
        }

        function handleFashionFile(file) {
            const reader = new FileReader();
            reader.onload = (e) => {
                fashionPreview.src = e.target.result;
                fashionPreview.classList.add('active');
            };
            reader.readAsDataURL(file);
        }

        // Form submission handling
        generationForm.addEventListener('submit', function(e) {
            e.preventDefault();
            const formData = new FormData(this);
            const genType = formData.get('gen_type');
            const isRefaceMode = genType === 'face' && formData.get('reface_mode') === 'on';
            const loadingOverlay = document.getElementById('loadingOverlay');
            const loadingTitle = document.getElementById('loadingTitle');
            const loadingStatus = document.getElementById('loadingStatus');
            const progressBar = document.getElementById('progressBar');

            // Hide all processing steps
            document.querySelectorAll('.processing-steps').forEach(steps => {
                steps.style.display = 'none';
            });

            // Show appropriate processing steps based on generation type
            let processingSteps;
            switch(genType) {
                case 'face':
                    loadingTitle.textContent = 'Generating Image';
                    processingSteps = document.getElementById('faceGenerationSteps');
                    break;
                case 'fashion':
                    loadingTitle.textContent = 'Creating Fashion Design';
                    processingSteps = document.getElementById('fashionDesignSteps');
                    break;
                case 'prompt':
                    loadingTitle.textContent = 'Generating Prompt';
                    processingSteps = document.getElementById('promptGenerationSteps');
                    break;
            }

            if (processingSteps) {
                processingSteps.style.display = 'block';
            }

            // Reset progress bar
            progressBar.style.width = '0%';

            // Validate form based on generation type
            if (genType === 'face') {
                if (isRefaceMode) {
                    // In reface mode, check for face and target image
                    const faceInput = document.getElementById('faceInput');
                    const faceSelect = document.querySelector('input[name="face_select"]');
                    const refaceTargetInput = document.getElementById('refaceTargetInput');

                    const hasFace = (faceInput.files.length > 0) || (faceSelect.value && faceSelect.value !== '');

                    if (!hasFace) {
                        alert('Please upload or select a face image');
                        return;
                    }

                    if (!refaceTargetInput.files.length) {
                        alert('Please upload a target image to apply your face to');
                        return;
                    }
                } else {
                    // Standard face generation mode
                    const textPrompt = document.querySelector('textarea[name="text_prompt"]');

                    if (!textPrompt.value.trim()) {
                        alert('Please enter a text prompt');
                        return;
                    }
                }
            } else if (genType === 'fashion') {
                const fashionInput = document.getElementById('fashionInput');
                const fashionPrompt = document.querySelector('textarea[name="fashion_prompt"]');

                if (!fashionInput.files.length) {
                    alert('Please upload a base image for fashion design');
                    return;
                }

                if (!fashionPrompt.value.trim()) {
                    alert('Please enter a design prompt');
                    return;
                }
            } else if (genType === 'prompt') {
                const promptInput = document.getElementById('promptInput');

                if (!promptInput.files.length) {
                    alert('Please upload an image for prompt generation');
                    return;
                }

                loadingOverlay.classList.add('active');
                loadingStatus.textContent = 'Initializing prompt generation...';

                const promptFormData = new FormData();
                promptFormData.append('prompt_image', promptInput.files[0]);
                promptFormData.append('top_k', document.getElementById('top_k').value);
                promptFormData.append('max_length', document.getElementById('max_length').value);
                promptFormData.append('min_length', document.getElementById('min_length').value);
                promptFormData.append('num_beams', document.getElementById('num_beams').value);

                // Update progress bar for prompt generation
                const steps = processingSteps.querySelectorAll('.processing-step');
                let currentStep = 0;

                const updateProgress = () => {
                    const progress = ((currentStep + 1) / steps.length) * 100;
                    progressBar.style.width = `${progress}%`;

                    // Update step status
                    steps.forEach((step, index) => {
                        step.classList.remove('active', 'completed', 'error');
                        if (index < currentStep) {
                            step.classList.add('completed');
                        } else if (index === currentStep) {
                            step.classList.add('active');
                        }
                    });
                };

                // Simulate step progression
                const stepDurations = {
                    'init': 2000,
                    'blip': 3000,
                    'clip': 2500,
                    'combine': 2000,
                    'finalize': 1500
                };

                const simulateSteps = () => {
                    const stepNames = ['init', 'blip', 'clip', 'combine', 'finalize'];
                    let currentIndex = 0;

                    const processStep = () => {
                        if (currentIndex < stepNames.length) {
                            const stepName = stepNames[currentIndex];
                            loadingStatus.textContent = steps[currentIndex].querySelector('.step-title').textContent;
                            currentStep = currentIndex;
                            updateProgress();

                            setTimeout(() => {
                                currentIndex++;
                                processStep();
                            }, stepDurations[stepName]);
                        }
                    };

                    processStep();
                };

                simulateSteps();

                fetch('/generate_prompt', {
                    method: 'POST',
                    body: promptFormData
                })
                .then(response => response.json())
                .then(data => {
                    loadingOverlay.classList.remove('active');
                    if (data.success) {
                        document.getElementById('generatedPrompt').value = data.prompt;
                    } else {
                        alert('Error generating prompt: ' + data.message);
                    }
                })
                .catch(error => {
                    loadingOverlay.classList.remove('active');
                    console.error('Error generating prompt:', error);
                    alert('Error generating prompt');
                });

                return;
            }

            loadingOverlay.classList.add('active');

            fetch(this.action, {
                method: 'POST',
                body: formData
            })
            .then(response => response.json())
            .then(data => {
                if (data.status === 'started' && data.process_id) {
                    const eventSource = new EventSource(`/process_status/${data.process_id}`);
                    const steps = processingSteps.querySelectorAll('.processing-step');
                    let currentStep = 0;

                    const updateProgress = () => {
                        const progress = ((currentStep + 1) / steps.length) * 100;
                        progressBar.style.width = `${progress}%`;
                    };

                    eventSource.onmessage = function(event) {
                        const data = JSON.parse(event.data);
                        loadingStatus.textContent = data.message;

                        if (data.step) {
                            steps.forEach((step, index) => {
                                const stepName = step.getAttribute('data-step');
                                step.classList.remove('active', 'completed', 'error');

                                if (stepName === data.step) {
                                    step.classList.add('active');
                                    currentStep = index;
                                    updateProgress();
                                } else if (data.completed_steps && data.completed_steps.includes(stepName)) {
                                    step.classList.add('completed');
                                }
                            });
                        }

                        if (data.status === 'completed') {
                            eventSource.close();
                            loadingOverlay.classList.remove('active');
                            window.location.reload();
                        } else if (data.status === 'error') {
                            eventSource.close();
                            loadingOverlay.classList.remove('active');
                            alert('Error: ' + data.message);
                        }
                    };

                    eventSource.onerror = function() {
                        eventSource.close();
                        loadingOverlay.classList.remove('active');
                        alert('Connection error occurred');
                    };
                } else {
                    loadingOverlay.classList.remove('active');
                    alert('Error starting process');
                }
            })
            .catch(error => {
                loadingOverlay.classList.remove('active');
                alert('Error submitting form: ' + error);
            });
        });

        // Modal functionality
        const modal = document.getElementById('imageModal');
        const modalImg = document.getElementById('modalImage');
        const closeBtn = document.querySelector('.modal-close');
        const prevBtn = document.getElementById('prevBtn');
        const nextBtn = document.getElementById('nextBtn');
        const imageCounter = document.getElementById('imageCounter');
        const fashionDesignBtn = document.getElementById('fashionDesignBtn');
        let currentImages = [];
        let currentImageIndex = 0;

        function updateModalImage(prompt) {
            // Show loading spinner
            const loadingSpinner = document.createElement('div');
            loadingSpinner.className = 'modal-image-loading';
            modalImg.parentElement.appendChild(loadingSpinner);

            // Reset image opacity
            modalImg.classList.remove('loaded');

            // Preload the image
            const image = currentImages[currentImageIndex];
            const tempImage = new Image();
            tempImage.onload = function() {
                modalImg.src = image.src;
                modalImg.classList.add('loaded');
                loadingSpinner.remove();
            };
            tempImage.src = image.src;

            document.getElementById('modalPrompt').textContent = `Prompt: ${prompt}`;

            // Update navigation buttons
            prevBtn.disabled = currentImageIndex === 0;
            nextBtn.disabled = currentImageIndex === currentImages.length - 1;

            // Update counter
            imageCounter.textContent = `${currentImageIndex + 1} / ${currentImages.length}`;

            // Show/hide fashion design button based on image type
            if (image.title && image.title.includes('Fashion Design')) {
                fashionDesignBtn.style.display = 'none';
            } else {
                fashionDesignBtn.style.display = 'block';
            }
        }

        // Preload images when modal opens
        document.querySelectorAll('.gallery-item').forEach(item => {
            item.addEventListener('click', function() {
                currentImages = JSON.parse(this.dataset.images);
                currentImageIndex = 0;

                // Preload all images
                currentImages.forEach(img => {
                    const tempImage = new Image();
                    tempImage.src = img.src;
                });

                updateModalImage(this.dataset.prompt);
                modal.classList.add('active');
            });
        });

        // Fashion design button click handler
        fashionDesignBtn.addEventListener('click', function() {
            // Close the modal
            modal.classList.remove('active');

            // Switch to fashion design tab
            document.querySelector('input[value="fashion"]').click();

            // Get the current image URL
            const currentImage = currentImages[currentImageIndex];
            const imageUrl = currentImage.src;

            // Create a new File object from the image URL
            fetch(imageUrl)
                .then(response => response.blob())
                .then(blob => {
                    const file = new File([blob], 'fashion_base.png', { type: 'image/png' });

                    // Create a new DataTransfer object
                    const dataTransfer = new DataTransfer();
                    dataTransfer.items.add(file);

                    // Set the file input's files
                    const fashionInput = document.getElementById('fashionInput');
                    fashionInput.files = dataTransfer.files;

                    // Trigger the file input change event
                    const event = new Event('change', { bubbles: true });
                    fashionInput.dispatchEvent(event);
                })
                .catch(error => {
                    console.error('Error loading image:', error);
                    alert('Error loading image for fashion design');
                });
        });

        prevBtn.addEventListener('click', () => {
            if (currentImageIndex > 0) {
                currentImageIndex--;
                updateModalImage(document.getElementById('modalPrompt').textContent.replace('Prompt: ', ''));
            }
        });

        nextBtn.addEventListener('click', () => {
            if (currentImageIndex < currentImages.length - 1) {
                currentImageIndex++;
                updateModalImage(document.getElementById('modalPrompt').textContent.replace('Prompt: ', ''));
            }
        });

        closeBtn.addEventListener('click', () => {
            modal.classList.remove('active');
        });

        modal.addEventListener('click', (e) => {
            if (e.target === modal) {
                modal.classList.remove('active');
            }
        });

        document.addEventListener('keydown', (e) => {
            if (e.key === 'Escape' && modal.classList.contains('active')) {
                modal.classList.remove('active');
            } else if (modal.classList.contains('active')) {
                if (e.key === 'ArrowLeft' && !prevBtn.disabled) {
                    prevBtn.click();
                } else if (e.key === 'ArrowRight' && !nextBtn.disabled) {
                    nextBtn.click();
                }
            }
        });

        // Download image button functionality
        document.getElementById('downloadImageBtn').addEventListener('click', function() {
            if (currentImages && currentImages.length > 0) {
                const currentImage = currentImages[currentImageIndex];

                // Create a temporary link element
                const downloadLink = document.createElement('a');
                downloadLink.href = currentImage.src;

                // Extract filename from the URL
                const filename = currentImage.src.split('/').pop();
                downloadLink.download = filename;

                // Append to body, click and remove
                document.body.appendChild(downloadLink);
                downloadLink.click();
                document.body.removeChild(downloadLink);

                // Show feedback
                const originalText = this.innerHTML;
                this.innerHTML = '<svg width="20" height="20" viewBox="0 0 20 20" fill="none" xmlns="http://www.w3.org/2000/svg"><path d="M8 15L3 10L4.41 8.59L8 12.17L15.59 4.58L17 6L8 15Z" fill="currentColor"/></svg>Downloaded';
                setTimeout(() => {
                    this.innerHTML = originalText;
                }, 2000);
            }
        });

        // Copy prompt button functionality
        document.getElementById('copyPromptBtn').addEventListener('click', function() {
            const promptText = document.getElementById('generatedPrompt');
            promptText.select();
            document.execCommand('copy');

            // Show feedback
            const originalText = this.innerHTML;
            this.innerHTML = '<svg width="20" height="20" viewBox="0 0 20 20" fill="none" xmlns="http://www.w3.org/2000/svg"><path d="M8 15L3 10L4.41 8.59L8 12.17L15.59 4.58L17 6L8 15Z" fill="currentColor"/></svg>Copied!';
            setTimeout(() => {
                this.innerHTML = originalText;
            }, 2000);
        });

        // Delete image functionality
        document.addEventListener('DOMContentLoaded', function() {
            // Attach event listeners to all delete buttons
            document.querySelectorAll('.gallery-item__delete').forEach(button => {
                button.addEventListener('click', function(e) {
                    e.stopPropagation(); // Prevent opening the modal

                    if (!confirm('Are you sure you want to delete this image? This cannot be undone.')) {
                        return;
                    }

                    const baseFile = this.getAttribute('data-base');
                    const finalFile = this.getAttribute('data-final');

                    // Show loading toast
                    showToast('Deleting image...', 'success', 'Processing');

                    // Send delete request
                    fetch('/delete_image', {
                        method: 'POST',
                        headers: {
                            'Content-Type': 'application/json',
                        },
                        body: JSON.stringify({
                            baseFile: baseFile,
                            finalFile: finalFile
                        })
                    })
                    .then(response => response.json())
                    .then(data => {
                        if (data.success) {
                            // Find and remove the gallery item from the DOM
                            const galleryItem = this.closest('.gallery-item');
                            if (galleryItem) {
                                galleryItem.style.opacity = '0';
                                galleryItem.style.transform = 'scale(0.8)';
                                galleryItem.style.transition = 'all 0.3s ease';

                                // Remove after animation
                                setTimeout(() => {
                                    galleryItem.remove();

                                    // Check if gallery is empty and update display
                                    const galleryItems = document.querySelectorAll('.gallery-item');
                                    if (galleryItems.length === 0) {
                                        const gallery = document.querySelector('.gallery-grid');
                                        if (gallery) {
                                            gallery.innerHTML = '<p class="text-muted">No generations yet.</p>';
                                        }
                                    }
                                }, 300);
                            }

                            showToast(data.message, 'success', 'Success');
                        } else {
                            showToast(data.message, 'error', 'Error');
                        }
                    })
                    .catch(error => {
                        showToast('An error occurred while deleting the image', 'error', 'Error');
                        console.error('Error:', error);
                    });
                });
            });
        });

        // Prompt upload handling
        const promptUpload = document.getElementById('promptUpload');
        const promptInput = document.getElementById('promptInput');
        const promptPreview = document.getElementById('promptPreview');

        if (promptUpload && promptInput) {
            promptUpload.addEventListener('click', () => {
                promptInput.click();
            });

            promptUpload.addEventListener('dragover', (e) => {
                e.preventDefault();
                promptUpload.style.borderColor = 'var(--primary-color)';
            });

            promptUpload.addEventListener('dragleave', () => {
                promptUpload.style.borderColor = 'var(--border-color)';
            });

            promptUpload.addEventListener('drop', (e) => {
                e.preventDefault();
                promptUpload.style.borderColor = 'var(--border-color)';
                const file = e.dataTransfer.files[0];
                if (file && file.type.startsWith('image/')) {
                    handlePromptFile(file);
                }
            });

            promptInput.addEventListener('change', (e) => {
                const file = e.target.files[0];
                if (file) {
                    handlePromptFile(file);
                }
            });
        }

        function handlePromptFile(file) {
            const reader = new FileReader();
            reader.onload = (e) => {
                promptPreview.src = e.target.result;
                promptPreview.classList.add('active');
            };
            reader.readAsDataURL(file);
        }

        // Function to show toast notifications
        function showToast(message, type = 'success', title = '') {
            const toastContainer = document.getElementById('toastContainer');
            if (!toastContainer) {
                console.error("Toast container not found!");
                return;
            }
            const toast = document.createElement('div');
            toast.className = `toast ${type}`;

            // Default titles based on type
            if (!title) {
                title = type === 'success' ? 'Success' : 'Error';
            }

            toast.innerHTML = `
                <div class="toast-icon">
                    ${type === 'success'
                        ? '<svg width="24" height="24" viewBox="0 0 24 24" fill="none" xmlns="http://www.w3.org/2000/svg"><path d="M9 16.17L4.83 12L3.41 13.41L9 19L21 7L19.59 5.59L9 16.17Z" fill="#10B981"/></svg>'
                        : '<svg width="24" height="24" viewBox="0 0 24 24" fill="none" xmlns="http://www.w3.org/2000/svg"><path d="M12 2C6.48 2 2 6.48 2 12C2 17.52 6.48 22 12 22C17.52 22 22 17.52 22 12C22 6.48 17.52 2 12 2ZM13 17H11V15H13V17ZM13 13H11V7H13V13Z" fill="#EF4444"/></svg>'}
                </div>
                <div class="toast-content">
                    <div class="toast-title">${title}</div>
                    <div class="toast-message">${message}</div>
                </div>
                <button class="toast-close">&times;</button>
            `;

            toastContainer.appendChild(toast);

            // Show the toast with a slight delay for animation
            setTimeout(() => {
                toast.classList.add('show');
            }, 10);

            // Add click handler for close button
            toast.querySelector('.toast-close').addEventListener('click', () => {
                toast.classList.remove('show');
                setTimeout(() => {
                    toastContainer.removeChild(toast);
                }, 300);
            });

            // Auto-hide after 5 seconds
            setTimeout(() => {
                if (toast.parentNode) {
                    toast.classList.remove('show');
                    setTimeout(() => {
                        if (toast.parentNode) {
                            toastContainer.removeChild(toast);
                        }
                    }, 300);
                }
            }, 5000);
        }

        // Flag to track if backup dialog has been shown
        let backupDialogShown = false;

        // Wait for DOM to be fully loaded before attaching event listeners
        document.addEventListener('DOMContentLoaded', function() {
            // Only check for backup automatically if gallery is empty
            const galleryItems = document.querySelectorAll('.gallery-item');
            if (galleryItems.length === 0) {
                console.log("Gallery is empty, checking for backup");
                checkForBackupAtStartup();
            } else {
                console.log("Gallery has items, skipping automatic backup check");
            }

            // Handle Google Drive backup
            const saveButton = document.querySelector('a[href="/save_to_drive"]');
            if (saveButton) {
                saveButton.addEventListener('click', function(e) {
                    e.preventDefault();
                    console.log("Save to Drive button clicked");

                    // Show loading toast
                    showToast('Saving to Google Drive...', 'success', 'Processing');

                    fetch('/save_to_drive')
                        .then(response => response.json())
                        .then(data => {
                            if (data.success) {
                                showToast(data.message, 'success', 'Backup Successful');
                            } else {
                                showToast(data.message, 'error', 'Backup Failed');
                            }
                        })
                        .catch(error => {
                            showToast('An error occurred while saving to Google Drive', 'error', 'Backup Failed');
                        });
                });
            } else {
                console.error("Save to Drive button not found");
            }

            // Handle Google Drive restore
            const restoreButton = document.querySelector('a[href="/restore_from_drive"]');
            if (restoreButton) {
                restoreButton.addEventListener('click', function(e) {
                    e.preventDefault();
                    console.log("Restore from Drive button clicked");

                    // Always show dialog for manual restore
                    backupDialogShown = false;

                    // Show loading toast
                    showToast('Checking for backups on Google Drive...', 'success', 'Processing');

                    // Explicitly check for backups when button is clicked
                    fetch('/check_for_backup')
                        .then(response => response.json())
                        .then(data => {
                            if (data.exists) {
                                // Show confirmation dialog
                                const backupDialog = document.getElementById('backupDialog');
                                if (backupDialog) {
                                    backupDialog.querySelector('.backup-dialog-message').textContent =
                                        `A backup was found in Google Drive: ${data.backup}. Would you like to restore it?`;
                                    backupDialog.classList.add('show');
                                } else {
                                    console.error("Backup dialog not found");
                                }
                            } else {
                                showToast(data.message, 'error', 'No Backup Found');
                            }
                        })
                        .catch(error => {
                            showToast('An error occurred while checking for backups', 'error', 'Error');
                        });
                });
            } else {
                console.error("Restore from Drive button not found");
            }

            // Handle backup dialog buttons
            const cancelBtn = document.getElementById('cancelRestoreBtn');
            if (cancelBtn) {
                cancelBtn.addEventListener('click', function() {
                    const backupDialog = document.getElementById('backupDialog');
                    if (backupDialog) {
                        backupDialog.classList.remove('show');
                    }
                    // Set flag to prevent showing again during this session
                    backupDialogShown = true;
                });
            }

            const confirmBtn = document.getElementById('confirmRestoreBtn');
            if (confirmBtn) {
                confirmBtn.addEventListener('click', function() {
                    const backupDialog = document.getElementById('backupDialog');
                    if (backupDialog) {
                        backupDialog.classList.remove('show');
                    }
                    // Set flag to prevent showing again
                    backupDialogShown = true;

                    // Show loading toast
                    showToast('Restoring from Google Drive...', 'success', 'Processing');

                    fetch('/restore_from_drive')
                        .then(response => response.json())
                        .then(data => {
                            if (data.success) {
                                showToast(data.message, 'success', 'Restore Successful');
                                // Reload the page after a short delay to show updated gallery
                                setTimeout(() => {
                                    window.location.reload();
                                }, 2000);
                            } else {
                                showToast(data.message, 'error', 'Restore Failed');
                            }
                        })
                        .catch(error => {
                            showToast('An error occurred while restoring from Google Drive', 'error', 'Restore Failed');
                        });
                });
            }
        });

        // Check for backup at startup
        function checkForBackupAtStartup() {
            // If dialog has already been shown, don't show it again
            if (backupDialogShown) return;

            console.log("Checking for backup at startup");
            fetch('/check_for_backup')
                .then(response => response.json())
                .then(data => {
                    if (data.exists) {
                        // Mark as shown to prevent duplicate dialogs
                        backupDialogShown = true;
                        console.log("Backup found:", data.backup);

                        // Show the dialog after a short delay
                        setTimeout(() => {
                            const backupDialog = document.getElementById('backupDialog');
                            if (backupDialog) {
                                backupDialog.querySelector('.backup-dialog-message').textContent =
                                    `A backup was found in Google Drive: ${data.backup}. Would you like to restore it?`;
                                backupDialog.classList.add('show');
                            }
                        }, 1000);
                    } else {
                        console.log("No backup found");
                    }
                })
                .catch(error => {
                    console.error('Error checking for backup at startup:', error);
                });
        }
    </script>

    <!-- Toast Container -->
    <div class="toast-container" id="toastContainer"></div>

    <!-- Backup Restore Dialog -->
    <div class="backup-dialog" id="backupDialog">
        <div class="backup-dialog-content">
            <div class="backup-dialog-title">Backup Found</div>
            <div class="backup-dialog-message">A previous backup was found in Google Drive. Would you like to restore it?</div>
            <div class="backup-dialog-actions">
                <button class="btn btn-secondary" id="cancelRestoreBtn">No, Thanks</button>
                <button class="btn btn-primary" id="confirmRestoreBtn">Yes, Restore Backup</button>
            </div>
        </div>
    </div>
</body>
</html>
"""

@app.route('/')
def index():
    # Filter out hidden files and directories from faces list
    faces = [f for f in sorted(os.listdir(UPLOAD_FOLDER))
             if not f.startswith('.') and not f.endswith('.ipynb') and not f.endswith('.ipynb_checkpoints')]

    # Filter out unwanted files from generated folder
    gens = [f for f in sorted(os.listdir(GENERATED_FOLDER))
            if not f.startswith('.')
            and not f.endswith('.ipynb')
            and not f.endswith('.ipynb_checkpoints')
            and not f.startswith('mask_')
            and not f.endswith('.prompt')]

    swaps = {os.path.splitext(f)[0]: f for f in os.listdir(SWAPPED_FOLDER)}
    history = []

    for g in gens:
        key = os.path.splitext(g)[0]

        # Read the prompt
        prompt = ''
        prompt_path = os.path.join(GENERATED_FOLDER, f"{key}.prompt")
        if os.path.exists(prompt_path):
            with open(prompt_path, 'r') as f:
                prompt = f.read().strip()

        # Check if this is a fashion design
        is_fashion = g.startswith('fashion_')

        if is_fashion:
            # For fashion designs, check if there's a face-swapped version
            timestamp = key.replace('fashion_', '')
            swapped_key = f"{timestamp}_final"

            if swapped_key in swaps:
                # If face swap exists, show both original and swapped
                history.append((g, swaps[swapped_key], prompt, True, True))  # Added is_swapped flag
            else:
                # If no face swap, show only the generated image
                history.append((g, None, prompt, True, False))
        elif key + "_final" in swaps:
            # For regular generations with face swap, show both original and swapped
            history.append((g, swaps[key + "_final"], prompt, False, True))
        else:
            # For "Generate Only" mode, show just the generated image
            history.append((g, None, prompt, False, False))

    return render_template_string(HTML, faces=faces, history=history)

# Global status tracking
status_queues = defaultdict(queue.Queue)
active_processes = {}

def update_status(process_id, message, step=None, completed_steps=None, status=None):
    """Update processing status for a specific process"""
    status_queues[process_id].put({
        'message': message,
        'step': step,
        'completed_steps': completed_steps,
        'status': status
    })

@app.route('/process_status/<process_id>')
def process_status(process_id):
    def generate():
        if process_id not in status_queues:
            yield f"data: {json.dumps({'status': 'error', 'message': 'Invalid process ID'})}\n\n"
            return

        while True:
            try:
                # Get status update from queue with increased timeout
                status = status_queues[process_id].get(timeout=300)  # Increased timeout to 5 minutes

                # Send the status update
                yield f"data: {json.dumps(status)}\n\n"

                # If this is a completion or error status, end the stream
                if status.get('status') in ['completed', 'error']:
                    # Cleanup
                    if process_id in status_queues:
                        del status_queues[process_id]
                    if process_id in active_processes:
                        del active_processes[process_id]
                    break

            except queue.Empty:
                # If no updates for 5 minutes, end the stream
                yield f"data: {json.dumps({'status': 'error', 'message': 'Processing timeout - operation taking longer than expected'})}\n\n"
                # Cleanup
                if process_id in status_queues:
                    del status_queues[process_id]
                if process_id in active_processes:
                    del active_processes[process_id]
                break

    return Response(generate(), mimetype='text/event-stream')

@app.route('/run_generation', methods=['POST'])
def run_generation():
    try:
        gen_type = request.form.get('gen_type', 'face')

        # Create a copy of form data as a dict for modification
        form_data = dict(request.form)

        # Handle file uploads before spawning the thread to avoid closed file issues
        face_path = None
        fashion_path = None
        reface_target_path = None

        # Check if we're in reface mode (direct face swap)
        is_reface_mode = gen_type == 'face' and form_data.get('reface_mode') == 'on'
        if is_reface_mode:
            form_data['is_reface_mode'] = True

        # Handle face upload if present
        if gen_type == 'face' and 'face_upload' in request.files:
            face_file = request.files['face_upload']
            if face_file and face_file.filename:
                # Create a safe filename
                filename = secure_filename(face_file.filename)
                face_path = os.path.join(UPLOAD_FOLDER, filename)

                # Save the file directly
                try:
                    face_file.save(face_path)
                    # Add the filename to form data
                    form_data['face_file_path'] = face_path
                except Exception as e:
                    return jsonify({'error': f"Error saving face file: {str(e)}"}), 500

        # Handle reface target upload if in reface mode
        if is_reface_mode and 'reface_target_upload' in request.files:
            target_file = request.files['reface_target_upload']
            if target_file and target_file.filename:
                # Create a safe filename
                filename = secure_filename(target_file.filename)
                reface_target_path = os.path.join(GENERATED_FOLDER, f"target_{filename}")

                # Save the file directly
                try:
                    target_file.save(reface_target_path)
                    # Add the filename to form data
                    form_data['reface_target_path'] = reface_target_path
                except Exception as e:
                    return jsonify({'error': f"Error saving target file: {str(e)}"}), 500

        # Handle fashion upload if present
        if gen_type == 'fashion' and 'fashion_upload' in request.files:
            fashion_file = request.files['fashion_upload']
            if fashion_file and fashion_file.filename:
                # Create a safe filename
                filename = secure_filename(fashion_file.filename)
                fashion_path = os.path.join(GENERATED_FOLDER, f"temp_{filename}")

                # Save the file directly
                try:
                    fashion_file.save(fashion_path)
                    # Add the filename to form data
                    form_data['fashion_file_path'] = fashion_path
                except Exception as e:
                    return jsonify({'error': f"Error saving fashion file: {str(e)}"}), 500

        # Generate unique process ID
        process_id = datetime.datetime.now().strftime("%Y%m%d-%H%M%S-%f")

        # Start processing in a separate thread without passing file objects
        thread = threading.Thread(target=process_generation, args=(process_id, gen_type, form_data, None))
        thread.daemon = True  # Make thread daemon so it exits when main thread exits
        thread.start()

        # Store thread reference
        active_processes[process_id] = thread

        return jsonify({'status': 'started', 'process_id': process_id})

    except Exception as e:
        return jsonify({'error': str(e)}), 500

def generate_mask(image):
    """Generate a mask with black background, black face, and white body."""
    # STEP 1: Detect the face first
    img_cv = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2BGR)
    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
    gray_cv = cv2.cvtColor(img_cv, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray_cv, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30))

    # Get face coordinates with padding
    face_box = None
    if len(faces) > 0:
        face = max(faces, key=lambda x: x[2] * x[3])
        x, y, w, h = face
        padding = int(max(w, h) * 0.5)
        x1 = max(0, x - padding)
        y1 = max(0, y - padding)
        x2 = min(img_cv.shape[1], x + w + padding)
        y2 = min(img_cv.shape[0], y + h + padding)
        face_box = (x1, y1, x2, y2)

    # STEP 2: Remove background to get person silhouette
    person_img = remove(image)
    person_np = np.array(person_img)
    gray = cv2.cvtColor(person_np, cv2.COLOR_RGBA2GRAY)
    _, binary = cv2.threshold(gray, 1, 255, cv2.THRESH_BINARY)

    # STEP 3: Create a black mask (background black)
    mask = Image.new('RGB', image.size, (0, 0, 0))

    # STEP 4: Draw the person silhouette in white
    person_mask_pil = Image.fromarray(binary)
    mask_draw = ImageDraw.Draw(mask)
    mask_draw.bitmap((0, 0), person_mask_pil, fill=(255, 255, 255))

    # STEP 5: Draw a black box over the face area
    if face_box:
        mask_draw.rectangle(face_box, fill=(0, 0, 0))

    return mask

# Add a new function for archiving generation results
def archive_generation(gen_type, timestamp, form_data, generation_time, face_path=None, base_image=None, final_image=None, prompt=None):
    """
    Archive generation results in a structured folder system

    Args:
        gen_type (str): Type of generation ('face', 'fashion', 'prompt')
        timestamp (str): Timestamp of the generation
        form_data (dict): Form data used for generation
        generation_time (float): Time taken to generate in seconds
        face_path (str, optional): Path to the face image if used
        base_image (str, optional): Path to the generated base image
        final_image (str, optional): Path to the final swapped image
        prompt (str, optional): The prompt used for generation
    """
    # Create archives directory if it doesn't exist
    archives_dir = "archives"
    os.makedirs(archives_dir, exist_ok=True)

    # Create a subfolder for this generation
    generation_folder = os.path.join(archives_dir, timestamp)
    os.makedirs(generation_folder, exist_ok=True)

    # Create and write the prompt.txt file with metadata
    with open(os.path.join(generation_folder, "prompt.txt"), "w") as f:
        f.write(f"Generation Type: {gen_type}\n")
        f.write(f"Timestamp: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"Duration: {generation_time:.2f} seconds\n")
        f.write("\nGeneration Parameters:\n")

        # Add specific parameters based on generation type
        if gen_type == "face":
            f.write(f"Body Type: {form_data.get('body_type', 'N/A')}\n")
            f.write(f"Breast Size: {form_data.get('breast_size', 'N/A')}\n")
            f.write(f"Hip Shape: {form_data.get('hip_shape', 'N/A')}\n")
            f.write(f"Other Traits: {form_data.get('other_traits', 'N/A')}\n")
        elif gen_type == "fashion":
            f.write(f"Restore Face: {form_data.get('restore_face', 'No')}\n")
        elif gen_type == "prompt":
            f.write(f"Top K Style Keywords: {form_data.get('top_k', '5')}\n")
            f.write(f"Max Length: {form_data.get('max_length', '50')}\n")
            f.write(f"Min Length: {form_data.get('min_length', '20')}\n")
            f.write(f"Number of Beams: {form_data.get('num_beams', '5')}\n")

        # Write the prompt used
        f.write("\nPrompt:\n")
        f.write(prompt if prompt else "No prompt available")

    # Copy all relevant files to the archive folder
    if face_path and os.path.exists(face_path):
        shutil.copy2(face_path, os.path.join(generation_folder, "face_image" + os.path.splitext(face_path)[1]))

    if base_image and os.path.exists(base_image):
        shutil.copy2(base_image, os.path.join(generation_folder, "generated_image" + os.path.splitext(base_image)[1]))

    # Copy any intermediate face swap iterations if they exist
    for i in range(1, NUM_SWAPS + 1):
        iter_path = os.path.join(SWAPPED_FOLDER, f"{timestamp}_iter{i}.png")
        if os.path.exists(iter_path):
            shutil.copy2(iter_path, os.path.join(generation_folder, f"iter{i}.png"))

    # Copy the final face-swapped image if it exists
    if final_image and os.path.exists(final_image):
        shutil.copy2(final_image, os.path.join(generation_folder, "final_image" + os.path.splitext(final_image)[1]))

    # For fashion design, copy the mask if it exists
    mask_path = os.path.join(GENERATED_FOLDER, f"mask_{timestamp}.png")
    if os.path.exists(mask_path):
        shutil.copy2(mask_path, os.path.join(generation_folder, "mask.png"))

    return generation_folder

def process_generation(process_id, gen_type, form_data, files):
    """Process generation request"""
    try:
        start_time = time.time()
        update_status(process_id, 'Initializing...', step='init')
        timestamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
        face_path = None
        base_image = None
        final_image = None
        prompt = ""

        # Handle reface mode (direct face swap without generation)
        if gen_type == 'face' and form_data.get('is_reface_mode'):
            update_status(process_id, 'Processing direct face swap...', step='init')

            # Get the face path
            face_path = form_data.get('face_file_path')
            if not face_path and form_data.get('face_select'):
                face_path = os.path.join(UPLOAD_FOLDER, form_data.get('face_select'))

            if not face_path or not os.path.exists(face_path):
                update_status(process_id, 'Error: No valid face image provided!', status='error')
                return

            # Get the target image path
            target_path = form_data.get('reface_target_path')
            if not target_path or not os.path.exists(target_path):
                update_status(process_id, 'Error: No valid target image provided!', status='error')
                return

            update_status(process_id, 'Applying face swap...', step='swap')

            # Save the target image to generated folder first
            base_name = f"reface_{timestamp}.png"
            base_path = os.path.join(GENERATED_FOLDER, base_name)
            shutil.copy2(target_path, base_path)
            base_image = base_path

            # Set a simple prompt for record-keeping
            prompt = "Direct face swap"

            # Save the prompt
            prompt_path = os.path.join(GENERATED_FOLDER, f"reface_{timestamp}.prompt")
            with open(prompt_path, 'w') as f:
                f.write(prompt)

            # Do face swap
            target = base_path
            key = f"reface_{timestamp}"

            for i in range(1, NUM_SWAPS + 1):
                temp = os.path.join(SWAPPED_FOLDER, f"{key}_iter{i}.png")

                # Use the dedicated face swap function
                success = perform_face_swap(
                    source_path=face_path,
                    target_path=target,
                    output_path=temp
                )

                if success:
                    target = temp
                    update_status(process_id, f'Face swap iteration {i} completed', step='swap')
                else:
                    update_status(process_id, f'Error: Face swap iteration {i} failed', status='error')
                    return

            # Save final result
            final = os.path.join(SWAPPED_FOLDER, f"{key}_final.png")
            final_image = final

            try:
                if os.path.exists(target):
                    shutil.copy2(target, final)
                    update_status(process_id, 'Final result saved successfully', step='enhance')
                else:
                    update_status(process_id, 'Error: Final face swap result not found', status='error')
                    return
            except Exception as e:
                update_status(process_id, f'Error saving final result: {str(e)}', status='error')
                return

            # Clean up the temporary target file
            if os.path.exists(target_path):
                os.remove(target_path)

            # Archive the generation
            generation_time = time.time() - start_time
            archive_generation(
                gen_type="reface",
                timestamp=timestamp,
                form_data=form_data,
                generation_time=generation_time,
                face_path=face_path,
                base_image=base_image,
                final_image=final_image,
                prompt=prompt
            )

            update_status(process_id, 'Face swap complete!', status='completed')
            return

        elif gen_type == 'fashion':
            # Handle fashion design using pre-saved file path
            fashion_path = form_data.get('fashion_file_path')
            if not fashion_path or not os.path.exists(fashion_path):
                update_status(process_id, 'Error: no valid image uploaded!', status='error')
                return

            update_status(process_id, 'Processing uploaded image...', step='base')

            # Get the prompt
            prompt = form_data.get('fashion_prompt', '')
            if not prompt:
                update_status(process_id, 'Error: no prompt provided!', status='error')
                return

            update_status(process_id, 'Processing image...', step='face')

            try:
                # Process the image
                with Image.open(fashion_path) as image:
                    image = square_image(image)

                    # Generate mask (preserve face and background)
                    mask = generate_mask(image)

                    update_status(process_id, 'Saving mask...', step='swap')

                    # Save the mask image
                    mask_path = os.path.join(GENERATED_FOLDER, f"mask_{timestamp}.png")
                    mask.save(mask_path)

                    # Expand the mask for smoother transitions
                    mask = expand_mask(mask)

                    # Convert images
                    image = image.convert("RGB")
                    mask = mask.convert("RGB")

                    update_status(process_id, 'Generating fashion design...', step='enhance')

                    # Generate the fashion design
                    result = inpainting_pipeline(
                        prompt=prompt,
                        image=image,
                        mask_image=mask,
                        num_inference_steps=80,
                        guidance_scale=7.5,
                        negative_prompt=NEG_PROMPT
                    ).images[0]

                    update_status(process_id, 'Saving result...', step='optimize')

                    # Save the fashion design
                    fashion_output = os.path.join(GENERATED_FOLDER, f"fashion_{timestamp}.png")
                    result.save(fashion_output)
                    base_image = fashion_output

                    # Save the prompt
                    prompt_path = os.path.join(GENERATED_FOLDER, f"fashion_{timestamp}.prompt")
                    with open(prompt_path, 'w') as f:
                        f.write(prompt)

                    # Handle face restoration if requested
                    if form_data.get('restore_face'):
                        update_status(process_id, 'Restoring face quality...', step='finalize')

                        # Use the original image as source for face swap
                        target = fashion_output
                        key = timestamp

                        for i in range(1, NUM_SWAPS + 1):
                            temp = os.path.join(SWAPPED_FOLDER, f"{key}_iter{i}.png")

                            # Use the dedicated face swap function
                            success = perform_face_swap(
                                source_path=fashion_path,
                                target_path=target,
                                output_path=temp
                            )

                            if success:
                                target = temp
                                update_status(process_id, f'Face restoration iteration {i} completed', step='finalize')
                            else:
                                update_status(process_id, f'Error: Face restoration iteration {i} failed', status='error')
                                return

                        final = os.path.join(SWAPPED_FOLDER, f"{key}_final.png")
                        shutil.copyfile(target, final)
                        final_image = final

            except Exception as e:
                update_status(process_id, f'Error processing fashion image: {str(e)}', status='error')
                return
            finally:
                # Cleanup temporary file
                if os.path.exists(fashion_path):
                    os.remove(fashion_path)

            # Archive the generation
            generation_time = time.time() - start_time
            archive_generation(
                gen_type=gen_type,
                timestamp=timestamp,
                form_data=form_data,
                generation_time=generation_time,
                face_path=fashion_path,
                base_image=base_image,
                final_image=final_image,
                prompt=prompt
            )

            update_status(process_id, 'Fashion design complete!', status='completed')
        else:
            # Handle regular generation
            update_status(process_id, 'Processing face selection...', step='init')

            # 1) Handle face selection - Now optional
            ffile = None
            if gen_type == 'face':
                # Check for pre-saved face file path
                face_path = form_data.get('face_file_path')
                if face_path and os.path.exists(face_path):
                    ffile = os.path.basename(face_path)
                else:
                    # Try to get from face_select
                    ffile = form_data.get('face_select')
                    if ffile:
                        face_path = os.path.join(UPLOAD_FOLDER, ffile)

            # 2) Build prompt
            txt = form_data.get('text_prompt', '')

            # Normalize and filter out 'none' values


            mods = ",".join(filter(
                lambda x: x and x.lower() != 'none',
                [
                    (form_data.get('body_type') or ''),
                    (form_data.get('breast_size') or ''),
                    (form_data.get('hip_shape') or ''),
                    (form_data.get('other_traits') or '').strip()
                ]
            ))

            prompt = f"{txt}, {mods}" if mods else txt


            update_status(process_id, 'Generating base image...', step='base')

            # 3) Generate image
            try:
                img = pipe(
                    prompt,
                    height=HEIGHT, width=WIDTH,
                    num_inference_steps=STEPS,
                    guidance_scale=GUIDANCE,
                    negative_prompt=NEG_PROMPT
                ).images[0]
                update_status(process_id, 'Base image generated successfully', step='base')
            except Exception as e:
                update_status(process_id, f'Error generating base image: {str(e)}', status='error')
                return

            base = f"{timestamp}.png"
            out1 = os.path.join(GENERATED_FOLDER, base)
            base_image = out1

            try:
                img.save(out1)
                update_status(process_id, 'Base image saved successfully', step='base')
            except Exception as e:
                update_status(process_id, f'Error saving generated image: {str(e)}', status='error')
                return

            # 4) Face swap if face is provided
            if gen_type == 'face' and face_path and os.path.exists(face_path):
                update_status(process_id, 'Processing face details...', step='face')
                target = out1
                key = timestamp

                update_status(process_id, 'Applying face swap...', step='swap')

                # Verify files exist before proceeding
                if not os.path.exists(target):
                    update_status(process_id, 'Error: Generated image not found', status='error')
                    return

                if not os.path.exists(face_path):
                    update_status(process_id, 'Error: Face image not found', status='error')
                    return

                for i in range(1, NUM_SWAPS + 1):
                    temp = os.path.join(SWAPPED_FOLDER, f"{key}_iter{i}.png")
                    try:
                        update_status(process_id, f'Running face swap iteration {i}...', step='swap')

                        # Use the dedicated face swap function
                        success = perform_face_swap(
                            source_path=face_path,
                            target_path=target,
                            output_path=temp
                        )

                        if success:
                            target = temp
                            update_status(process_id, f'Face swap iteration {i} completed', step='swap')
                        else:
                            update_status(process_id, f'Error: Face swap iteration {i} failed', status='error')
                            return

                    except Exception as e:
                        update_status(process_id, f'Error during face swap: {str(e)}', status='error')
                        return

                update_status(process_id, 'Enhancing final result...', step='enhance')
                final = os.path.join(SWAPPED_FOLDER, f"{key}_final.png")
                final_image = final

                try:
                    if os.path.exists(target):
                        shutil.copy2(target, final)
                        update_status(process_id, 'Final result saved successfully', step='enhance')
                    else:
                        update_status(process_id, 'Error: Final face swap result not found', status='error')
                        return
                except Exception as e:
                    update_status(process_id, f'Error saving final result: {str(e)}', status='error')
                    return

            update_status(process_id, 'Saving prompt...', step='optimize')
            # Save the prompt
            prompt_path = os.path.join(GENERATED_FOLDER, f"{timestamp}.prompt")
            try:
                with open(prompt_path, 'w') as f:
                    f.write(prompt)
                update_status(process_id, 'Prompt saved successfully', step='optimize')
            except Exception as e:
                update_status(process_id, f'Error saving prompt: {str(e)}', status='error')
                return

            # Archive the generation
            generation_time = time.time() - start_time
            archive_generation(
                gen_type=gen_type,
                timestamp=timestamp,
                form_data=form_data,
                generation_time=generation_time,
                face_path=face_path,
                base_image=base_image,
                final_image=final_image,
                prompt=prompt
            )

            update_status(process_id, 'Generation complete!', status='completed')

    except Exception as e:
        update_status(process_id, f'Unexpected error: {str(e)}', status='error')

@app.route('/generated/<path:filename>')
def serve_generated(filename):
    return send_from_directory(GENERATED_FOLDER, filename)

@app.route('/swapped/<path:filename>')
def serve_swapped(filename):
    return send_from_directory(SWAPPED_FOLDER, filename)

@app.route('/download_pdf')
def download_pdf_report():
    pdf_path = "report.pdf"
    doc = SimpleDocTemplate(pdf_path, pagesize=letter)
    styles = getSampleStyleSheet()

    # Update styles
    styles['Title'].fontSize = 24
    styles['Title'].spaceAfter = 30
    styles['Title'].textColor = colors.HexColor('#4f46e5')

    styles['Heading1'].fontSize = 16
    styles['Heading1'].textColor = colors.HexColor('#4f46e5')
    styles['Heading1'].spaceAfter = 12

    styles['Normal'].fontSize = 10
    styles['Normal'].textColor = colors.HexColor('#6b7280')
    styles['Normal'].spaceAfter = 12

    story = []

    # Title
    story.append(Paragraph("AI Character Generation Report", styles['Title']))
    story.append(Paragraph("Generated Results & Analysis", styles['Normal']))
    story.append(Spacer(1, 20))

    # Get all generations
    gens = sorted(os.listdir(GENERATED_FOLDER))
    swaps = {os.path.splitext(f)[0]: f for f in os.listdir(SWAPPED_FOLDER)}

    for i, gen in enumerate(gens):
        if i > 0:
            story.append(PageBreak())

        key = os.path.splitext(gen)[0]
        if key + "_final" in swaps:
            # Create image comparison table
            table_data = []

            # Add original and final images
            orig = os.path.join(GENERATED_FOLDER, gen)
            fin = os.path.join(SWAPPED_FOLDER, swaps[key + "_final"])

            if os.path.exists(orig) and os.path.exists(fin):
                img1 = Image(orig, width=250, height=250)
                img2 = Image(fin, width=250, height=250)
                table_data.append([
                    Paragraph("Original Generated", styles['Normal']),
                    Paragraph("Face Swapped", styles['Normal'])
                ])
                table_data.append([img1, img2])

            if table_data:
                table = Table(table_data, colWidths=[250, 250])
                table.setStyle(TableStyle([
                    ('ALIGN', (0, 0), (-1, -1), 'CENTER'),
                    ('VALIGN', (0, 0), (-1, -1), 'MIDDLE'),
                    ('GRID', (0, 0), (-1, -1), 1, colors.HexColor('#e2e8f0')),
                    ('BACKGROUND', (0, 0), (-1, 0), colors.HexColor('#f5f3ff')),
                    ('TEXTCOLOR', (0, 0), (-1, 0), colors.HexColor('#4f46e5')),
                    ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
                    ('FONTSIZE', (0, 0), (-1, 0), 10),
                    ('BOTTOMPADDING', (0, 0), (-1, 0), 12),
                    ('TOPPADDING', (0, 0), (-1, 0), 12),
                ]))
                story.append(table)

        story.append(Spacer(1, 20))

    doc.build(story)
    return send_file(pdf_path, as_attachment=True)

@app.route('/download_zip')
def download_zip():
    import tempfile

    # Create a temporary directory
    temp_dir = tempfile.mkdtemp()

    try:
        # Create subdirectories
        original_dir = os.path.join(temp_dir, 'original_images')
        final_dir = os.path.join(temp_dir, 'final_images')

        os.makedirs(original_dir)
        os.makedirs(final_dir)

        # Copy original and final images
        gens = sorted(os.listdir(GENERATED_FOLDER))
        swaps = {os.path.splitext(f)[0]: f for f in os.listdir(SWAPPED_FOLDER)}

        for gen in gens:
            key = os.path.splitext(gen)[0]
            if key + "_final" in swaps:
                orig_path = os.path.join(GENERATED_FOLDER, gen)
                final_path = os.path.join(SWAPPED_FOLDER, swaps[key + "_final"])

                if os.path.exists(orig_path):
                    shutil.copy2(orig_path, os.path.join(original_dir, gen))
                if os.path.exists(final_path):
                    shutil.copy2(final_path, os.path.join(final_dir, swaps[key + "_final"]))

        # Create ZIP file
        zip_path = os.path.join(temp_dir, 'generated_images.zip')
        with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
            # Add original and final images
            for root, _, files in os.walk(temp_dir):
                for file in files:
                    if file.endswith('.zip'):
                        continue
                    file_path = os.path.join(root, file)
                    arcname = os.path.relpath(file_path, temp_dir)
                    zipf.write(file_path, arcname)

        return send_file(zip_path, as_attachment=True, download_name='generated_images.zip')

    finally:
        # Clean up temporary directory
        shutil.rmtree(temp_dir)

@app.route('/faces/<path:filename>')
def serve_face(filename):
    return send_from_directory(UPLOAD_FOLDER, filename)

@app.route('/fashion_design', methods=['POST'])
def fashion_design():
    try:
        data = request.json
        image_url = data['image_url']
        prompt = data['prompt']

        # Get the image path from the URL
        if 'generated' in image_url:
            image_path = os.path.join(GENERATED_FOLDER, os.path.basename(image_url))
        else:
            image_path = os.path.join(SWAPPED_FOLDER, os.path.basename(image_url))

        if not os.path.exists(image_path):
            return jsonify({'error': 'Image not found'}), 404

        # Load and process the image
        image = Image.open(image_path)
        image = square_image(image)

        # Generate mask (inverse of face mask - we want to modify everything except the face)
        face_mask = generate_mask(image)
        mask = ImageOps.invert(face_mask)

        # Save the mask image
        mask_path = os.path.join(GENERATED_FOLDER, f"mask_{datetime.datetime.now().strftime('%Y%m%d-%H%M%S')}.png")
        mask.save(mask_path)

        # Expand the mask for smoother transitions
        mask = expand_mask(mask)

        # Convert images to the correct format
        image = image.convert("RGB")
        mask = mask.convert("RGB")

        # Generate random seed for reproducibility
        seed = torch.randint(0, 2**32, (1,)).item()

        if torch.cuda.is_available():
            generator = torch.Generator("cuda").manual_seed(seed)
        else:
            generator = torch.Generator().manual_seed(seed)

        # Generate the fashion design
        result = inpainting_pipeline(
            prompt=prompt,
            image=image,
            mask_image=mask,
            generator=generator,
            num_inference_steps=100,
            guidance_scale=7.5,
            strength=0.8,
            negative_prompt=NEG_PROMPT
        ).images[0]

        # Save the result
        timestamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
        output_path = os.path.join(GENERATED_FOLDER, f"fashion_{timestamp}.png")
        result.save(output_path)

        # Save the prompt
        prompt_path = output_path + '.prompt'
        with open(prompt_path, 'w') as f:
            f.write(prompt)

        return jsonify({
            'success': True,
            'image': url_for('serve_generated', filename=os.path.basename(output_path))
        })

    except Exception as e:
        print(f"Error in fashion design: {str(e)}")  # Add logging
        return jsonify({'error': str(e)}), 500

# Add these helper functions after the imports and before the app initialization
def square_image(image):
    """Make the image square by padding it."""
    width, height = image.size
    size = max(width, height)
    new_image = Image.new('RGB', (size, size), (255, 255, 255))
    new_image.paste(image, ((size - width) // 2, (size - height) // 2))
    return new_image

def detect_face(image):
    """Detect face in the image using OpenCV."""
    # Convert PIL image to OpenCV format
    img_cv = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2BGR)

    # Load face detection model
    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

    # Convert to grayscale
    gray = cv2.cvtColor(img_cv, cv2.COLOR_BGR2GRAY)

    # Detect faces with adjusted parameters for better detection
    faces = face_cascade.detectMultiScale(
        gray,
        scaleFactor=1.1,
        minNeighbors=5,
        minSize=(30, 30),
        flags=cv2.CASCADE_SCALE_IMAGE
    )

    if len(faces) > 0:
        # Get the largest face
        face = max(faces, key=lambda x: x[2] * x[3])
        x, y, w, h = face
        # Add more padding to ensure we capture the entire face
        padding = int(max(w, h) * 0.3)  # Increased padding
        x1 = max(0, x - padding)
        y1 = max(0, y - padding)
        x2 = min(img_cv.shape[1], x + w + padding)
        y2 = min(img_cv.shape[0], y + h + padding)
        return (x1, y1, x2, y2)
    return None

def expand_mask(mask, expand_pixels=20):
    """Expand the mask by a few pixels to ensure smooth transitions."""
    mask_array = np.array(mask)
    kernel = np.ones((expand_pixels, expand_pixels), np.uint8)
    expanded = cv2.dilate(mask_array, kernel, iterations=1)
    return Image.fromarray(expanded)

@app.route('/favicon.ico')
def favicon():
    # Create a simple favicon on-the-fly
    img = Image.new('RGB', (32, 32), color = (73, 109, 137))
    img_io = io.BytesIO()
    img.save(img_io, 'PNG')
    img_io.seek(0)
    return send_file(img_io, mimetype='image/png')

# Function to check and setup roop if needed
def setup_roop():
    """Setup roop directory and dependencies if not already present"""
    global ROOP_DIR

    # Use absolute path for better reliability
    ROOP_DIR = os.path.abspath("roop")

    # Check if CUDA is available
    has_cuda = torch.cuda.is_available()
    if has_cuda:
        print("CUDA is available, using GPU acceleration")
        gpu_name = torch.cuda.get_device_name(0)
        print(f"GPU: {gpu_name}")
    else:
        print("CUDA is not available, will use CPU mode")

    if not os.path.exists(ROOP_DIR):
        print(f"Roop directory not found. Creating at {ROOP_DIR}")
        # Clone roop repository
        os.system("git clone https://github.com/s0md3v/roop.git")

        # Install requirements with CUDA support
        os.system("pip install onnx insightface")

        # Install appropriate onnxruntime version based on CUDA availability
        if has_cuda:
            os.system("pip install onnxruntime-gpu")
        else:
            os.system("pip install onnxruntime")

        os.system("pip install -r roop/requirements.txt")

        # Create needed directories
        os.makedirs(os.path.join(ROOP_DIR, "models"), exist_ok=True)

        # Download necessary model files
        os.system("wget -nc https://huggingface.co/henryruhs/roop/resolve/main/inswapper_128.onnx -O roop/models/inswapper_128.onnx")

        print("Roop setup completed")
    else:
        print(f"Roop directory found at {ROOP_DIR}")

    # Verify model file exists
    model_path = os.path.join(ROOP_DIR, "models", "inswapper_128.onnx")
    if not os.path.exists(model_path):
        print(f"Model file not found at {model_path}. Downloading...")
        os.makedirs(os.path.dirname(model_path), exist_ok=True)
        os.system(f"wget -nc https://huggingface.co/henryruhs/roop/resolve/main/inswapper_128.onnx -O {model_path}")

# Function to check CUDA availability
def check_cuda_availability():
    """Check if CUDA is available for GPU acceleration."""
    try:
        if torch.cuda.is_available():
            device_count = torch.cuda.device_count()
            device_name = torch.cuda.get_device_name(0) if device_count > 0 else "Unknown"
            print(f"CUDA is available with {device_count} device(s). Using: {device_name}")
            return True
        else:
            print("CUDA is not available. Using CPU.")
            return False
    except:
        print("Error checking CUDA. Defaulting to CPU.")
        return False

# Function to execute face swap
def perform_face_swap(source_path, target_path, output_path):
    """Perform face swap using roop with Colab T4 GPU compatibility."""
    # Ensure paths exist
    if not os.path.exists(source_path) or not os.path.exists(target_path):
        return False

    # Ensure output directory exists
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    # Face swap command optimized for Colab T4 GPU
    cmd = (
        f"python {ROOP_DIR}/run.py"
        f" --source \"{source_path}\""
        f" --target \"{target_path}\""
        f" -o \"{output_path}\""
        f" --execution-provider cuda"
        f" --frame-processor face_swapper face_enhancer"
        f" --many-faces"
        f" --reference-face-position 0"
        f" --similar-face-distance 0.85"
    )

    result = os.system(cmd)
    if result != 0:
        # If first attempt fails, try without CUDA
        cmd = (
            f"python {ROOP_DIR}/run.py"
            f" --source \"{source_path}\""
            f" --target \"{target_path}\""
            f" -o \"{output_path}\""
            f" --execution-provider cpu"
            f" --frame-processor face_swapper face_enhancer"
            f" --many-faces"
            f" --reference-face-position 0"
            f" --similar-face-distance 0.85"
        )
        result = os.system(cmd)

    return os.path.exists(output_path)

# Remove the custom face enhancement function as we're using roop's built-in enhancer
def enhance_face_quality(image_path):
    """Legacy function, now returns True since we're using roop's enhancer"""
    return True

# Add a new function to process generated images for improved quality
def post_process_image(image_path):
    """Apply post-processing to improve the overall image quality"""
    try:
        # Load the image
        img = Image.open(image_path)

        # Convert to numpy array for OpenCV processing
        img_cv = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)

        # Apply a slight sharpening to the entire image
        sharpen_kernel = np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]])
        img_cv = cv2.filter2D(img_cv, -1, sharpen_kernel)

        # Apply color enhancement
        hsv = cv2.cvtColor(img_cv, cv2.COLOR_BGR2HSV)
        h, s, v = cv2.split(hsv)

        # Increase saturation slightly
        s = cv2.add(s, 10)

        # Increase brightness slightly
        v = cv2.add(v, 5)

        # Merge and convert back
        hsv = cv2.merge([h, s, v])
        img_cv = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)

        # Convert back to PIL Image and save
        enhanced_img = Image.fromarray(cv2.cvtColor(img_cv, cv2.COLOR_BGR2RGB))
        enhanced_img.save(image_path)

        return True
    except Exception as e:
        print(f"Error in post-processing: {str(e)}")
        return False

# Call setup function to ensure roop is ready
setup_roop()

@app.route('/generate_prompt', methods=['POST'])
def generate_prompt():
    try:
        if 'prompt_image' not in request.files:
            return jsonify({'success': False, 'message': 'No image uploaded'})

        file = request.files['prompt_image']
        if file.filename == '':
            return jsonify({'success': False, 'message': 'No image selected'})

        # Save the uploaded image temporarily
        temp_path = os.path.join(GENERATED_FOLDER, f"temp_{secure_filename(file.filename)}")
        file.save(temp_path)

        try:
            # Get generation parameters
            top_k = int(request.form.get('top_k', 5))
            max_length = int(request.form.get('max_length', 50))
            min_length = int(request.form.get('min_length', 20))
            num_beams = int(request.form.get('num_beams', 5))

            # Generate prompt
            prompt = image_to_prompt(
                temp_path,
                top_k=top_k,
                max_length=max_length,
                min_length=min_length,
                num_beams=num_beams
            )

            return jsonify({
                'success': True,
                'prompt': prompt
            })

        finally:
            # Clean up temporary file
            if os.path.exists(temp_path):
                os.remove(temp_path)

    except Exception as e:
        return jsonify({'success': False, 'message': str(e)})

# 2. Style keywords pool
STYLE_KEYWORDS = [
    "cinematic lighting", "bokeh", "high detail", "vibrant colors",
    "matte finish", "film grain", "soft shadows", "ultra-realistic",
    "dramatic contrast", "surreal atmosphere", "minimalist", "ethereal",
    "fantasy art", "digital painting", "watercolor style"
]

def image_to_prompt(
    image_path: str,
    top_k: int = 5,
    # ==== Generation controls ====
    max_length: int = 50,         # maximum tokens to generate :contentReference[oaicite:1]{index=1}
    min_length: int = 20,         # minimum tokens to generate
    num_beams: int = 5,           # beam search width
    length_penalty: float = 1.0,  # >1.0 favors longer outputs
    no_repeat_ngram_size: int = 2,# blocks repeated n-grams
    early_stopping: bool = True   # stop when EOS token is reached
) -> str:
    """
    Given an image file path, returns a rich prompt string:
      [BLIP caption (tunable length)] + [top-K style keywords via CLIP similarity].
    """
    # a) Load & preprocess image
    image = Image.open(image_path).convert("RGB")

    # b) Generate caption with BLIP
    blip_inputs = blip_processor(images=image, return_tensors="pt")
    caption_ids = blip_model.generate(
        **blip_inputs,
        max_length=max_length,
        min_length=min_length,
        num_beams=num_beams,
        length_penalty=length_penalty,
        no_repeat_ngram_size=no_repeat_ngram_size,
        early_stopping=early_stopping
    )
    caption = blip_processor.decode(caption_ids[0], skip_special_tokens=True)

    # c) Compute image embedding via CLIP
    clip_inputs = clip_processor(images=image, return_tensors="pt")
    with torch.no_grad():
        img_embed = clip_model.get_image_features(**clip_inputs)
    img_embed = F.normalize(img_embed, dim=-1)

    # d) Embed style keywords and rank by cosine similarity
    tokenized = clip_tokenizer(STYLE_KEYWORDS, padding=True, return_tensors="pt")
    with torch.no_grad():
        txt_embeds = clip_model.get_text_features(**tokenized)
    txt_embeds = F.normalize(txt_embeds, dim=-1)
    sims = (txt_embeds @ img_embed.T).squeeze(1)
    top_indices = sims.topk(top_k).indices.tolist()
    style_tokens = [STYLE_KEYWORDS[i] for i in top_indices]

    # e) Build and return final prompt
    prompt = f"{caption}, " + ", ".join(style_tokens)
    return prompt

@app.route('/delete_image', methods=['POST'])
def delete_image():
    """Delete generated and swapped images"""
    try:
        data = request.get_json()
        base_file = data.get('baseFile')
        final_file = data.get('finalFile')

        deleted_files = []

        # Delete the base generated file if it exists
        if base_file and base_file != "undefined" and base_file != "null":
            base_path = os.path.join(GENERATED_FOLDER, base_file)
            if os.path.exists(base_path) and os.path.isfile(base_path):
                os.remove(base_path)
                deleted_files.append(base_file)

            # Delete any associated prompt file
            base_name = os.path.splitext(base_file)[0]
            prompt_file = f"{base_name}.prompt"
            prompt_path = os.path.join(GENERATED_FOLDER, prompt_file)
            if os.path.exists(prompt_path):
                os.remove(prompt_path)
                deleted_files.append(prompt_file)

            # Delete any associated mask file
            mask_file = f"mask_{base_name}.png"
            mask_path = os.path.join(GENERATED_FOLDER, mask_file)
            if os.path.exists(mask_path):
                os.remove(mask_path)
                deleted_files.append(mask_file)

        # Delete the swapped file if it exists
        if final_file and final_file != "undefined" and final_file != "null":
            final_path = os.path.join(SWAPPED_FOLDER, final_file)
            if os.path.exists(final_path) and os.path.isfile(final_path):
                os.remove(final_path)
                deleted_files.append(final_file)

            # Delete any intermediate swapped files
            if "_final" in final_file:
                key = final_file.replace("_final.png", "")
                for i in range(1, NUM_SWAPS + 1):
                    iter_file = f"{key}_iter{i}.png"
                    iter_path = os.path.join(SWAPPED_FOLDER, iter_file)
                    if os.path.exists(iter_path):
                        os.remove(iter_path)
                        deleted_files.append(iter_file)

        if deleted_files:
            return jsonify({
                'success': True,
                'message': f'Successfully deleted {len(deleted_files)} file(s)',
                'deleted': deleted_files
            })
        else:
            return jsonify({
                'success': False,
                'message': 'No files found to delete'
            })

    except Exception as e:
        return jsonify({
            'success': False,
            'message': f'Error deleting files: {str(e)}'
        })

if __name__ == "__main__":
    # Initialize Google Drive
    initialize_google_drive()

    # Set up ngrok tunnel
    NGROK_AUTH_TOKEN = "2x0x5HHuUV5Ss7drpzJLbW3s7oa_6prh4t1osTXX22gLMYY2P"
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)

    # Start an HTTP tunnel on port 5000
    public_url = ngrok.connect(5000).public_url
    print(f" * ngrok tunnel running at: {public_url}")

    # Run the Flask app
    app.run(host='0.0.0.0', port=5000)
